In [ ]:
import os, sys, re, warnings
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)

os.environ["CUDA_VISIBLE_DEVICES"] = ""      # force CPU
device = torch.device("cpu")
print(f"PyTorch {torch.__version__} | device: {device}")


# %% Cell 2 — Fragment definitions & constants ------------------------------
fragment_seqs = {
    1:  "SRGVSRGGSRGARGLM",  2:  "SRGARGLMNGYRGPAN",
    3:  "NGYRGPANGFRGGYDG",  4:  "GFRGGYDGYRPSFSNT",
    5:  "YRPSFSNTPNSGYTQS",  6:  "PNSGYTQSQFSAPRDY",
    7:  "QFSAPRDYSGYQRDGY",  8:  "SGYQRDGYQQNFKRGS",
    9:  "QQNFKRGSGQSGPRGA",  10: "GQSGPRGAPRGRGGPP",
    11: "PRGRGGPPRPNRGMPQ",  12: "RPNRGMPQMNTQQVN",   # last fragment is len 15
}

FRAGMENT_LENGTH = 16
OVERLAP         = 8
STRIDE          = FRAGMENT_LENGTH - OVERLAP           # = 8

aa_order  = ['A','C','D','E','F','G','H','I','K','L',
             'M','N','P','Q','R','S','T','V','W','Y']
aa_to_idx = {aa: i for i, aa in enumerate(aa_order)}

charge_lookup = {'R': +1.0, 'K': +1.0, 'H': +1.0, 'D': -1.0, 'E': -1.0}

three_to_one = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}

expected_unique = STRIDE * (len(fragment_seqs) - 1) + len(fragment_seqs[12])
print(f"Stride {STRIDE} | expected unique residues N = {expected_unique}")


# %% Cell 3 — Global residue index map (verified correct) -------------------
global_to_info = {}
for fid, seq in fragment_seqs.items():
    g_start = STRIDE * (fid - 1)
    for local_i, aa in enumerate(seq):
        g = g_start + local_i
        if g not in global_to_info:
            global_to_info[g] = {'aa': aa, 'frags': [],
                                 'local_in_frag': {}, 'frag_length': {}}
        # consistency: if this residue was already seen (overlap), AA must match
        assert global_to_info[g]['aa'] == aa, (
            f"AA conflict at global {g}: {global_to_info[g]['aa']} vs {aa} (frag {fid})")
        global_to_info[g]['frags'].append(fid)
        global_to_info[g]['local_in_frag'][fid] = local_i
        global_to_info[g]['frag_length'][fid]   = len(seq)

global_to_info = dict(sorted(global_to_info.items()))
N = len(global_to_info)
assert list(global_to_info.keys()) == list(range(N)), "global indices not contiguous"
print(f"N = {N} residues (contiguous 0..{N-1}, all overlaps consistent)")


# %% Cell 4 — Overlap audit (verified correct) ------------------------------
all_ok = True
for fid in range(1, 12):
    shared = sorted(g for g, info in global_to_info.items()
                    if fid in info['frags'] and fid + 1 in info['frags'])
    aa_from_k = ''.join(global_to_info[g]['aa'] for g in shared)
    ok = (aa_from_k == fragment_seqs[fid][-OVERLAP:] == fragment_seqs[fid + 1][:OVERLAP]
          and len(shared) == OVERLAP)
    all_ok &= ok
assert all_ok, "overlap audit failed — fix before continuing"
print("Overlap audit: all 11 consecutive pairs share 8 matching residues ✓")


# %% Cell 5 — Load + SYMMETRIZE contact matrices  # ===== FIXED =============
# ---------------------------------------------------------------------------
# The raw CSV matrices are ASYMMETRIC (M[i,j] != M[j,i]). A residue-residue
# contact is physically symmetric, so we symmetrize BEFORE dropping zeros.
#
#   WHY before? Your original dropped zeros first, so a one-sided pair (a, 0)
#   kept value `a`, while a two-sided pair (a, b) got mean(a, b). That is an
#   inconsistent hybrid: ~89% of nonzero pairs (the one-sided ones) came out
#   ~2x too high relative to the two-sided ones. Symmetrizing first makes
#   every pair consistent.
#
# SYMMETRIZE_MODE:
#   'mean' : 0.5*(M[i,j] + M[j,i])   <- default. Use if the two directions are
#            noisy measurements of the SAME underlying contact frequency.
#   'max'  : max(M[i,j], M[j,i])     <- use if a contact should count fully
#            whenever EITHER direction observed it.
#   >>> Confirm which matches how these MD matrices were generated. <<<
# ---------------------------------------------------------------------------
SYMMETRIZE_MODE = 'mean'          # 'mean' or 'max'
DATA_DIR = Path("fragment_matrices-include-3-type-interactions")        # <-- point this at your CSV folder
# DATA_DIR pattern alt: f"{fid}fragment/prob_chain/out_Res_Res_5A/..." etc.


def parse_residue_label(label: str):
    """'PRO13' -> (13, 'P'). Returns (local_1based:int, one_letter:str) or None."""
    m_num = re.search(r'(\d+)\s*$', str(label))
    m_aa  = re.match(r'\s*([A-Za-z]{3})', str(label))
    if m_num is None or m_aa is None:
        return None
    return int(m_num.group(1)), three_to_one.get(m_aa.group(1).upper(), '?')


def load_symmetrized_fragment(csv_path: Path, fid: int, mode: str) -> pd.DataFrame:
    """Read one fragment matrix, validate labels, symmetrize, return long df
    ['res_i','res_j','prob'] of nonzero *directed* entries (i!=j)."""
    raw = pd.read_csv(csv_path, index_col=0)
    raw.index   = raw.index.astype(str).str.strip()
    raw.columns = raw.columns.astype(str).str.strip()

    # -- structural checks (catch pandas dup-renaming / mis-shaped files) --
    assert raw.index.is_unique,   f"frag {fid}: duplicate row labels"
    assert raw.columns.is_unique, f"frag {fid}: duplicate column labels"
    assert list(raw.index) == list(raw.columns), \
        f"frag {fid}: row labels != column labels (matrix not alignable)"

    raw = raw.apply(pd.to_numeric, errors='coerce').fillna(0.0)

    # -- label validation: number -> amino acid must match fragment sequence
    #    (this is what catches a wrong numbering convention or wrong file) --
    seq = fragment_seqs[fid]
    for label in raw.index:
        parsed = parse_residue_label(label)
        assert parsed is not None, f"frag {fid}: unparsable label {label!r}"
        loc1, aa1 = parsed
        assert 1 <= loc1 <= len(seq), \
            f"frag {fid}: label {label!r} local pos {loc1} outside 1..{len(seq)}"
        assert aa1 == seq[loc1 - 1], (
            f"frag {fid}: label {label!r} says {aa1} but sequence has "
            f"{seq[loc1 - 1]} at local {loc1} — wrong file or numbering")

    # -- report + symmetrize ----------------------------------------------
    M = raw.values.astype(np.float64)
    asym = float(np.abs(M - M.T).max())
    if mode == 'mean':
        M = 0.5 * (M + M.T)
    elif mode == 'max':
        M = np.maximum(M, M.T)
    else:
        raise ValueError(f"bad SYMMETRIZE_MODE {mode!r}")
    np.fill_diagonal(M, 0.0)                       # remove self-contacts here
    assert np.allclose(M, M.T), f"frag {fid}: symmetrization failed"

    sym = pd.DataFrame(M, index=raw.index, columns=raw.columns)
    long = sym.stack().reset_index()
    long.columns = ['res_i', 'res_j', 'prob']
    long = long[long['prob'] != 0.0].reset_index(drop=True)
    long.attrs['asym_before'] = asym
    return long


prob_h = {}
print("Loading + symmetrizing fragment matrices "
      f"(mode='{SYMMETRIZE_MODE}') ...")
for fid in range(1, 13):
    csv_path = DATA_DIR / f"fragment_{fid}_saltbridge_pipi_cationpi_matrix.csv"
    if not csv_path.exists():
        print(f"  [{fid:>2}] NOT FOUND: {csv_path}")
        prob_h[fid] = pd.DataFrame(columns=['res_i', 'res_j', 'prob'])
        continue
    long = load_symmetrized_fragment(csv_path, fid, SYMMETRIZE_MODE)
    prob_h[fid] = long
    print(f"  [{fid:>2}] {csv_path.name:<38} "
          f"asym_before={long.attrs['asym_before']:.2e} -> "
          f"{len(long):>4} nonzero directed entries")
print(f"Total nonzero directed contacts: {sum(len(v) for v in prob_h.values()):,}")


# %% Cell 6 — Node feature matrix [N x 23] (verified correct) ---------------
node_feats = np.zeros((N, 23), dtype=np.float32)
for g, info in global_to_info.items():
    aa = info['aa']
    node_feats[g, aa_to_idx[aa]] = 1.0                              # [0:20] one-hot
    node_feats[g, 20] = float(charge_lookup.get(aa, 0.0))          # [20]   charge
    node_feats[g, 21] = g / (N - 1)                                # [21]   global pos
    local_norms = [li / (fl - 1)                                   # [22]   mean local pos
                   for li, fl in zip(info['local_in_frag'].values(),
                                     info['frag_length'].values())]
    node_feats[g, 22] = float(np.mean(local_norms))

x = torch.tensor(node_feats, dtype=torch.float)
assert torch.all(x[:, :20].sum(1) == 1.0), "one-hot block malformed"
print(f"Node features x: {tuple(x.shape)}  "
      f"(charge vals {sorted(set(x[:,20].tolist()))})")


# %% Cell 7 — Build edges + graph  # ===== FIXED (now on symmetric data) ====
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, remove_self_loops

edges_src, edges_dst, edges_prob = [], [], []
pair_sources = defaultdict(list)

for fid, df in prob_h.items():
    frag_len = len(fragment_seqs[fid])
    for _, row in df.iterrows():
        m_i = re.search(r'(\d+)$', str(row['res_i']))
        m_j = re.search(r'(\d+)$', str(row['res_j']))
        if m_i is None or m_j is None:
            print(f"  ✗ unparsable label in frag {fid} — skipped")
            continue
        local_i, local_j = int(m_i.group(1)) - 1, int(m_j.group(1)) - 1
        if not (0 <= local_i < frag_len and 0 <= local_j < frag_len):
            continue
        g_i = STRIDE * (fid - 1) + local_i
        g_j = STRIDE * (fid - 1) + local_j
        if not (0 <= g_i < N and 0 <= g_j < N):
            print(f"  ✗ out-of-bounds {g_i},{g_j} (frag {fid}) — skipped")
            continue
        edges_src.append(g_i); edges_dst.append(g_j); edges_prob.append(row['prob'])
        if g_i != g_j:
            pair_sources[tuple(sorted((g_i, g_j)))].append((fid, row['prob']))

edge_index_raw = torch.tensor([edges_src, edges_dst], dtype=torch.long)
edge_attr_raw  = torch.tensor(edges_prob, dtype=torch.float32)
edge_index_raw, edge_attr_raw = remove_self_loops(edge_index_raw, edge_attr_raw)

# Contact graph (kept for reference / evaluation — NOT used for message passing)
edge_index_u, edge_attr_u = to_undirected(
    edge_index_raw, edge_attr_raw, num_nodes=N, reduce='mean')

# Merged undirected contact value per pair. On symmetrized data every pair's
# two directions are equal, so this mean is exact (no more one-sided inflation).
all_edges    = {pair: float(np.mean([p for _, p in srcs]))
                for pair, srcs in pair_sources.items()}
all_edge_set = set(all_edges.keys())

# solidity check: the three views of the contact set must agree
E_undir = edge_index_u.shape[1] // 2
assert E_undir == len(pair_sources) == len(all_edge_set), (
    f"edge bookkeeping mismatch: to_undirected={E_undir}, "
    f"pair_sources={len(pair_sources)}, all_edges={len(all_edge_set)}")

# bridge / fragment-membership node metadata
bridge_mask     = torch.zeros(N, dtype=torch.bool)
frag_membership = torch.zeros(N, 12, dtype=torch.bool)
for g, info in global_to_info.items():
    if len(info['frags']) > 1:
        bridge_mask[g] = True
    for fid in info['frags']:
        frag_membership[g, fid - 1] = True

# LABEL-INDEPENDENT backbone graph (i, i+1) for message passing.
# Using the contact graph here would leak the labels into the adjacency.
_bb_src, _bb_dst = [], []
for i in range(N - 1):
    _bb_src += [i, i + 1]
    _bb_dst += [i + 1, i]
edge_index_backbone = torch.tensor([_bb_src, _bb_dst], dtype=torch.long)
edge_attr_backbone  = torch.ones((edge_index_backbone.shape[1], 1), dtype=torch.float32)

data = Data(x=x, edge_index=edge_index_backbone,
            edge_attr=edge_attr_backbone, num_nodes=N)
data.bridge_mask     = bridge_mask
data.frag_membership = frag_membership
data = data.to(device)

print(f"Graph: x={tuple(data.x.shape)} | backbone edges={data.edge_index.shape[1]} "
      f"| contact pairs={E_undir} | bridge nodes={int(bridge_mask.sum())}")


# %% Cell 8 — Pair inventory (verified correct) -----------------------------
all_possible_pairs = set(combinations(range(N), 2))

md_observed_pairs = set()
for fid, seq in fragment_seqs.items():
    idxs = range(STRIDE * (fid - 1), STRIDE * (fid - 1) + len(seq))
    md_observed_pairs.update(tuple(sorted(p)) for p in combinations(idxs, 2))

positive_pairs = sorted(all_edge_set)                       # nonzero, MD-observed
zero_pairs     = sorted(md_observed_pairs - all_edge_set)   # true zeros
unknown_pairs  = sorted(all_possible_pairs - md_observed_pairs)  # never co-observed

assert len(positive_pairs) + len(zero_pairs) == len(md_observed_pairs)
assert len(md_observed_pairs) + len(unknown_pairs) == len(all_possible_pairs)
assert not (set(positive_pairs) & set(zero_pairs))
print(f"Pairs: total={len(all_possible_pairs)} | positive={len(positive_pairs)} "
      f"| true-zero={len(zero_pairs)} | unknown(predict)={len(unknown_pairs)}")


# %% Cell 9 — Assemble TRAINING-READY tensors  # ===== FIXED / CONSOLIDATED =
# ---------------------------------------------------------------------------
# Supervised set = every MD-observed pair (label is known):
#     positive pairs -> symmetric contact value ;  true-zero pairs -> 0.0
# Unknown (cross-fragment) pairs have NO label -> prediction-only set.
#
# For each pair we store the ordered (i, j) with i < j. Your training loop
# runs the GNN on `data`, gets node embeddings H, then predicts a value for
# each pair from (H[i], H[j]) and compares to `sup_target`.
# ---------------------------------------------------------------------------
positive_prob = {p: float(all_edges[p]) for p in positive_pairs}

supervised_pairs   = positive_pairs + zero_pairs
supervised_targets = np.asarray(
    [positive_prob[p] for p in positive_pairs] + [0.0] * len(zero_pairs),
    dtype=np.float32)
supervised_is_pos  = np.asarray(
    [True] * len(positive_pairs) + [False] * len(zero_pairs), dtype=bool)

# tensors -----------------------------------------------------------------
sup_pair_index = torch.tensor(np.asarray(supervised_pairs).T,
                              dtype=torch.long, device=device)   # [2, P]
sup_target     = torch.tensor(supervised_targets, device=device)  # [P]
sup_is_pos     = torch.tensor(supervised_is_pos, device=device)   # [P]
unknown_pair_index = torch.tensor(np.asarray(unknown_pairs).T,
                                  dtype=torch.long, device=device)  # [2, U]

# stratified train/val split (keeps positive:zero ratio in both splits) ---
_idx = np.arange(len(supervised_pairs))
_tr, _va = train_test_split(_idx, test_size=0.10, random_state=42,
                            stratify=supervised_is_pos)
train_idx = torch.tensor(_tr, dtype=torch.long, device=device)
val_idx   = torch.tensor(_va, dtype=torch.long, device=device)

# target stats (values are tiny ~1e-4 — see note in the chat about scaling) 
pos_vals = supervised_targets[supervised_is_pos]
print("Training-ready:")
print(f"  supervised pairs : {sup_pair_index.shape[1]}  "
      f"(train {train_idx.numel()} / val {val_idx.numel()})")
print(f"  positives        : {int(sup_is_pos.sum())}  "
      f"| zeros: {int((~sup_is_pos).sum())}")
print(f"  unknown to predict: {unknown_pair_index.shape[1]}")
print(f"  positive target range: [{pos_vals.min():.3e}, {pos_vals.max():.3e}]  "
      f"mean {pos_vals.mean():.3e}  median {np.median(pos_vals):.3e}")


In [ ]:
# CELL — Normalize targets by TRAIN-set global max (leakage-safe, wired in)
print("=" * 70)
print("Normalizing targets by train-set global maximum")
print("=" * 70)

# --- training-set positive mask (no val leakage into the scale constant) ---
train_mask = torch.zeros(len(supervised_pairs), dtype=torch.bool, device=device)
train_mask[train_idx] = True
train_pos_mask = train_mask & sup_is_pos

p_max = float(sup_target[train_pos_mask].max())
assert p_max > 0, "train-set max is 0 — no positive pairs in training split?"
print(f"p_max (train positives only) : {p_max:.12f}")

# --- normalized FULL target vector (zeros stay exactly 0) -------------------
# Keep raw sup_target for reporting metrics on the true scale.
sup_target_norm = sup_target / p_max          # [P]; use THIS as the train target

# inspection dict (same constant), plus note any val positive above the
# train max — normalizes to >1, which is expected and fine.
positive_prob_norm = {p: float(all_edges[p] / p_max) for p in positive_pairs}

nv = sup_target_norm[sup_is_pos]
n_over_1 = int((nv > 1.0).sum())
print(f"\nAfter normalization (positives): "
      f"min {nv.min():.6f} | mean {nv.mean():.6f} | max {nv.max():.6f}")
if n_over_1:
    print(f"  note: {n_over_1} val positive(s) exceed the train max -> >1 "
          f"(expected; clamp preds or leave as-is)")
print(f"  zeros remain exactly 0: {bool((sup_target_norm[~sup_is_pos] == 0).all())}")


In [ ]:
# CELL — Binary-classification pair split with distance-matched zeros
# Purpose:
#   Build a balanced binary contact/non-contact dataset:
#       positives = MD nonzero pairs
#       zeros     = true MD-zero pairs
#
# Important:
#   This is a SEPARATE classification split from the previous regression
#   train_idx / val_idx split. Do not mix the two split systems.

from sklearn.model_selection import train_test_split
from collections import defaultdict
import numpy as np
import torch

print("=" * 70)
print("Binary classification split: positives + distance-matched zeros")
print("=" * 70)

# -------------------------------------------------------------------
# 1. Positive pairs
# -------------------------------------------------------------------
positive_pairs_all = sorted(positive_prob_norm.keys())

pos_train, pos_test = train_test_split(
    positive_pairs_all,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print(f"Positive pairs total : {len(positive_pairs_all)}")
print(f"Positive train       : {len(pos_train)}")
print(f"Positive test        : {len(pos_test)}")

# -------------------------------------------------------------------
# 2. Zero-pair pool
#    True MD-zero pairs only; remove any accidental positive overlap.
# -------------------------------------------------------------------
zero_pool = sorted(set(zero_pairs) - set(positive_pairs_all))

print(f"Zero-pair pool       : {len(zero_pool)}")

assert len(zero_pool) > 0, "No zero pairs available."
assert len(set(zero_pool) & set(positive_pairs_all)) == 0, \
    "Positive/zero overlap detected."

# -------------------------------------------------------------------
# 3. Helper functions for sequence-distance matching
# -------------------------------------------------------------------
def pair_distance(pair):
    i, j = pair
    return abs(i - j)


def distance_bin(pair):
    """
    Bin pairs by sequence separation.
    For these 16-residue fragment matrices, most observed pairs have
    small-to-moderate distances. Binning prevents the classifier from
    learning a trivial distance shortcut.
    """
    d = pair_distance(pair)

    if d <= 2:
        return "01_very_short"
    elif d <= 4:
        return "02_short"
    elif d <= 8:
        return "03_medium"
    elif d <= 12:
        return "04_long"
    else:
        return "05_very_long"


def summarize_distances(name, pairs):
    d = np.array([pair_distance(p) for p in pairs], dtype=float)
    print(
        f"  {name:<14} n={len(pairs):>4} | "
        f"mean={d.mean():6.2f} | median={np.median(d):5.1f} | "
        f"min={int(d.min()):>2} | max={int(d.max()):>2}"
    )


def sample_distance_matched_zeros(pos_pairs, zero_candidates, rng):
    """
    Sample one zero pair per positive pair, approximately matched by
    sequence-distance bin.

    If a bin does not contain enough zeros, the function samples the
    remaining zeros from unused zero pairs globally. This avoids crashing
    while still preserving as much distance matching as possible.
    """
    zero_by_bin = defaultdict(list)
    for zp in zero_candidates:
        zero_by_bin[distance_bin(zp)].append(zp)

    pos_by_bin = defaultdict(list)
    for pp in pos_pairs:
        pos_by_bin[distance_bin(pp)].append(pp)

    sampled = []
    used = set()

    # First pass: sample from the same distance bin
    for b, ppairs in pos_by_bin.items():
        need = len(ppairs)
        available = [z for z in zero_by_bin[b] if z not in used]

        if len(available) >= need:
            chosen_idx = rng.choice(len(available), size=need, replace=False)
            chosen = [available[i] for i in chosen_idx]
        else:
            chosen = available

        sampled.extend(chosen)
        used.update(chosen)

    # Second pass: if some bins were short, fill from any unused zero pair
    remaining_need = len(pos_pairs) - len(sampled)

    if remaining_need > 0:
        unused = [z for z in zero_candidates if z not in used]
        assert remaining_need <= len(unused), (
            f"Not enough unused zero pairs. Need {remaining_need}, "
            f"but only {len(unused)} remain."
        )
        fill_idx = rng.choice(len(unused), size=remaining_need, replace=False)
        fill = [unused[i] for i in fill_idx]
        sampled.extend(fill)
        used.update(fill)

    assert len(sampled) == len(pos_pairs), \
        f"Expected {len(pos_pairs)} zeros, got {len(sampled)}."
    assert len(set(sampled)) == len(sampled), \
        "Duplicate zero pairs sampled."

    return sampled


# -------------------------??????????????????????????????????***************************??
# 4. Sample distance-matched zeros without train/test leakage
# ---------------------------------??????????????????????????????????***************************??
# rng = np.random.default_rng(42)

# # First sample train zeros from full zero pool
# zero_train = sample_distance_matched_zeros(
#     pos_pairs=pos_train,
#     zero_candidates=zero_pool,
#     rng=rng
# )

# # Remove train zeros before sampling test zeros
# zero_pool_for_test = sorted(set(zero_pool) - set(zero_train))

# zero_test = sample_distance_matched_zeros(
#     pos_pairs=pos_test,
#     zero_candidates=zero_pool_for_test,
#     rng=rng
# )



# -------------------------------------------------------------------
# 4. Use ALL zero pairs, without balancing positives and zeros
# -------------------------------------------------------------------
rng = np.random.default_rng(42)

zero_train, zero_test = train_test_split(
    zero_pool,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print("\nUsing all available positives and all available zeros:")
print(f"  Positives train/test : {len(pos_train)} / {len(pos_test)}")
print(f"  Zeros train/test     : {len(zero_train)} / {len(zero_test)}")


# -------------------------------------------------------------------
# 5. Leakage checks
# -------------------------------------------------------------------
assert len(set(pos_train) & set(pos_test)) == 0
assert len(set(zero_train) & set(zero_test)) == 0

assert len(set(pos_train) & set(zero_train)) == 0
assert len(set(pos_train) & set(zero_test)) == 0
assert len(set(pos_test)  & set(zero_train)) == 0
assert len(set(pos_test)  & set(zero_test)) == 0

assert len(set(train_pair if False else [])) == 0 if False else True  # harmless no-op

print("\nLeakage checks:")
print(f"  pos_train ∩ pos_test   : {len(set(pos_train) & set(pos_test))}")
print(f"  zero_train ∩ zero_test : {len(set(zero_train) & set(zero_test))}")
print(f"  any pos/zero overlap   : 0")

# -------------------------------------------------------------------
# 6. Build train/test pair lists and binary labels
# -------------------------------------------------------------------
train_pairs = list(pos_train) + list(zero_train)
test_pairs  = list(pos_test)  + list(zero_test)

y_train_cls = np.array(
    [1] * len(pos_train) + [0] * len(zero_train),
    dtype=np.int64
)

y_test_cls = np.array(
    [1] * len(pos_test) + [0] * len(zero_test),
    dtype=np.int64
)

# Shuffle within each split so positives/zeros are mixed
train_perm = rng.permutation(len(train_pairs))
test_perm  = rng.permutation(len(test_pairs))

train_pairs = [train_pairs[i] for i in train_perm]
y_train_cls = y_train_cls[train_perm]

test_pairs = [test_pairs[i] for i in test_perm]
y_test_cls = y_test_cls[test_perm]

# -------------------------------------------------------------------
# 7. Convert to PyTorch tensors for GNN pair classifier
# -------------------------------------------------------------------
train_pair_index = torch.tensor(
    np.asarray(train_pairs).T,
    dtype=torch.long,
    device=device
)  # [2, n_train_pairs]

test_pair_index = torch.tensor(
    np.asarray(test_pairs).T,
    dtype=torch.long,
    device=device
)  # [2, n_test_pairs]

y_train_cls_t = torch.tensor(
    y_train_cls,
    dtype=torch.float32,
    device=device
)  # [n_train_pairs]

y_test_cls_t = torch.tensor(
    y_test_cls,
    dtype=torch.float32,
    device=device
)  # [n_test_pairs]

# -------------------------------------------------------------------
# 8. Leakage-safe normalized contact strengths for positives only
#    Useful later if you add a second-stage regressor.
#    Scale is computed only from pos_train, not pos_test.
# -------------------------------------------------------------------
p_max_cls = max(float(all_edges[p]) for p in pos_train)
assert p_max_cls > 0.0, "p_max_cls is zero."

train_pos_values_norm = {
    pair: float(all_edges[pair] / p_max_cls)
    for pair in pos_train
}

test_pos_values_norm = {
    pair: float(all_edges[pair] / p_max_cls)
    for pair in pos_test
}

# -------------------------------------------------------------------
# 9. Distance distribution diagnostics
# -------------------------------------------------------------------
print("\nSequence-distance diagnostics:")
summarize_distances("pos_train", pos_train)
summarize_distances("zero_train", zero_train)
summarize_distances("pos_test", pos_test)
summarize_distances("zero_test", zero_test)

# -------------------------------------------------------------------
# 10. Final summary
# -------------------------------------------------------------------
print("\nFinal binary split summary:")
print(f"  Train pairs total    : {len(train_pairs)}")
print(f"    positives          : {int((y_train_cls == 1).sum())}")
print(f"    zeros              : {int((y_train_cls == 0).sum())}")
print(f"  Test pairs total     : {len(test_pairs)}")
print(f"    positives          : {int((y_test_cls == 1).sum())}")
print(f"    zeros              : {int((y_test_cls == 0).sum())}")

print("\nTensor shapes:")
print(f"  train_pair_index     : {tuple(train_pair_index.shape)}")
print(f"  test_pair_index      : {tuple(test_pair_index.shape)}")
print(f"  y_train_cls_t        : {tuple(y_train_cls_t.shape)}")
print(f"  y_test_cls_t         : {tuple(y_test_cls_t.shape)}")
print(f"  p_max_cls            : {p_max_cls:.12e}")

print("\nSample train pairs:")
for i in range(min(10, len(train_pairs))):
    print(f"  {train_pairs[i]} -> label {y_train_cls[i]}")

print("\nSample test pairs:")
for i in range(min(10, len(test_pairs))):
    print(f"  {test_pairs[i]} -> label {y_test_cls[i]}")

In [ ]:
# CELL — GNN model for binary residue-pair interaction classification
#
# This model uses:
#   1. A label-independent backbone graph for message passing.
#   2. Symmetric pair features for undirected residue-pair classification.
#   3. Logit output for BCEWithLogitsLoss.
#
# Expected pair_index shape:
#   pair_index: LongTensor [2, B]
#   Example: train_pair_index, test_pair_index from the previous split cell.

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATv2Conv, PairNorm


class ResiduePairGNN(nn.Module):
    """
    GNN for binary residue-pair interaction classification.

    Input:
        data.x          : [N, node_dim]
        data.edge_index : [2, E_backbone]
        data.edge_attr  : [E_backbone, 1]

    Important:
        data.edge_index is the label-independent backbone graph, not the
        MD contact graph. This avoids contact-label leakage.

    Pair prediction:
        Given residue pairs (i, j), predict whether the MD contact
        probability is nonzero:
            class 1 = positive/nonzero contact
            class 0 = true MD-zero pair
    """

    def __init__(
        self,
        node_dim=23,
        edge_dim=1,
        hidden_dim=96,
        heads=4,
        dropout=0.4
    ):
        super().__init__()

        assert hidden_dim % heads == 0, (
            f"hidden_dim={hidden_dim} must be divisible by heads={heads}"
        )

        self.hidden_dim = hidden_dim
        self.dropout = dropout

        # Project raw node features to hidden space for residual connection
        self.input_proj = nn.Linear(node_dim, hidden_dim)

        # GATv2 layer 1
        self.gnn1 = GATv2Conv(
            in_channels=node_dim,
            out_channels=hidden_dim // heads,
            heads=heads,
            concat=True,
            edge_dim=edge_dim,
            dropout=dropout
        )
        self.norm1 = PairNorm()

        # GATv2 layer 2
        self.gnn2 = GATv2Conv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // heads,
            heads=heads,
            concat=True,
            edge_dim=edge_dim,
            dropout=dropout
        )
        self.norm2 = PairNorm()

        # Symmetric pair feature:
        #   h_i + h_j        : symmetric combined embedding
        #   |h_i - h_j|      : symmetric difference
        #   h_i * h_j        : symmetric interaction
        #   sep_norm, inv_sep: sequence-separation features
        #
        # Total dimension = 3 * hidden_dim + 2
        pair_dim = 3 * hidden_dim + 2

        self.classifier = nn.Sequential(
            nn.Linear(pair_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, 1)  # binary-classification logits
        )

    def encode_nodes(self, x, edge_index, edge_attr):
        """
        Compute node embeddings.

        Args:
            x          : [N, node_dim]
            edge_index : [2, E]
            edge_attr  : [E, 1]

        Returns:
            h          : [N, hidden_dim]
        """
        x0 = self.input_proj(x)

        h = self.gnn1(x, edge_index, edge_attr)
        h = self.norm1(h)
        h = F.elu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.gnn2(h, edge_index, edge_attr)
        h = self.norm2(h)

        # Residual fusion
        h = h + x0
        h = F.elu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        return h

    def _unpack_pair_index(self, pair_index):
        """
        Accept pair_index in PyG-style shape [2, B].

        Also supports [B, 2] defensively, but the recommended format
        for this pipeline is [2, B].
        """
        if pair_index.dim() != 2:
            raise ValueError(
                f"pair_index must be 2D, got shape {tuple(pair_index.shape)}"
            )

        # Preferred shape from previous cell: [2, B]
        if pair_index.shape[0] == 2:
            i = pair_index[0]
            j = pair_index[1]

        # Defensive fallback: [B, 2]
        elif pair_index.shape[1] == 2:
            i = pair_index[:, 0]
            j = pair_index[:, 1]

        else:
            raise ValueError(
                "pair_index must have shape [2, B] or [B, 2], "
                f"got {tuple(pair_index.shape)}"
            )

        return i.long(), j.long()

    def make_pair_features(self, h, pair_index):
        """
        Build symmetric pair features for undirected residue pairs.

        Recommended pair_index shape:
            [2, B]

        Returns:
            pair_feat: [B, 3*hidden_dim + 2]
        """
        i, j = self._unpack_pair_index(pair_index)

        h_i = h[i]
        h_j = h[j]

        # Sequence separation |i-j|
        sep = torch.abs(i - j).float()
        N = h.shape[0]

        sep_norm = (sep / (N - 1)).unsqueeze(1)      # [B, 1], range [0, 1]
        inv_sep  = (1.0 / (sep + 1.0)).unsqueeze(1)  # [B, 1]

        # Symmetric pair representation
        pair_feat = torch.cat(
            [
                h_i + h_j,
                torch.abs(h_i - h_j),
                h_i * h_j,
                sep_norm,
                inv_sep,
            ],
            dim=1
        )

        return pair_feat

    def forward(self, data, pair_index):
        """
        Args:
            data       : PyG Data object
            pair_index : LongTensor [2, B] or [B, 2]

        Returns:
            logits     : [B]
        """
        h = self.encode_nodes(data.x, data.edge_index, data.edge_attr)
        pair_feat = self.make_pair_features(h, pair_index)
        logits = self.classifier(pair_feat).squeeze(-1)
        return logits

    @torch.no_grad()
    def predict_proba(self, data, pair_index):
        """
        Return sigmoid probabilities for class 1.
        """
        self.eval()
        logits = self.forward(data, pair_index)
        return torch.sigmoid(logits)

In [ ]:
model = ResiduePairGNN(
    node_dim=data.x.shape[1],          # 23
    edge_dim=data.edge_attr.shape[1],  # 1
    hidden_dim=96,
    heads=4,
    dropout=0.4
).to(device)

print(model)

# Sanity-check forward pass before training
model.eval()
with torch.no_grad():
    test_logits = model(data, train_pair_index[:, :8])

print(f"Sanity logits shape: {tuple(test_logits.shape)}")
print(f"Sanity logits      : {test_logits.detach().cpu().numpy()}")
assert test_logits.shape == (8,), "Forward-pass output shape is wrong."

In [ ]:
criterion = nn.BCEWithLogitsLoss()


# Loss for imbalance data

# positive_weight = 0.6
# zero_weight = 0.4

# criterion_base = nn.BCEWithLogitsLoss(reduction="none")

# def criterion(logits, targets):
#     """
#     Weighted BCEWithLogitsLoss.
    
#     targets:
#         1 -> positive pair, weight 0.6
#         0 -> zero pair, weight 0.4
#     """
#     targets = targets.float()

#     per_sample_loss = criterion_base(logits, targets)

#     sample_weights = torch.where(
#         targets == 1.0,
#         torch.tensor(positive_weight, dtype=torch.float32, device=targets.device),
#         torch.tensor(zero_weight, dtype=torch.float32, device=targets.device),
#     )

#     return (per_sample_loss * sample_weights).mean()

In [ ]:
# CELL — Binary classification evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss
)
import numpy as np


def evaluate_binary(y_true, y_prob, threshold=0.5):
    """
    Evaluate binary residue-pair classification.

    Args:
        y_true    : array-like [N], true labels 0/1
        y_prob    : array-like [N], predicted probabilities from sigmoid
        threshold : probability threshold for binary prediction

    Returns:
        dict of scalar metrics
    """

    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob).astype(float).reshape(-1)

    assert y_true.shape == y_prob.shape, (
        f"Shape mismatch: y_true {y_true.shape}, y_prob {y_prob.shape}"
    )

    assert np.all(np.isfinite(y_prob)), "y_prob contains NaN or inf."

    if np.any((y_prob < 0.0) | (y_prob > 1.0)):
        raise ValueError(
            "y_prob must contain probabilities in [0, 1]. "
            "Did you forget to apply sigmoid to logits?"
        )

    assert set(np.unique(y_true)).issubset({0, 1}), \
        f"y_true must contain only 0/1 labels, got {np.unique(y_true)}"

    # Binary predictions
    y_pred = (y_prob >= threshold).astype(int)

    # Force 2x2 confusion matrix even if one class is absent
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    # Core threshold-based metrics
    acc = accuracy_score(y_true, y_pred)

    sn = recall_score(
        y_true, y_pred,
        pos_label=1,
        zero_division=0
    )  # sensitivity / recall / TPR

    sp = tn / (tn + fp + 1e-12)  # specificity / TNR

    pr = precision_score(
        y_true, y_pred,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_true, y_pred,
        pos_label=1,
        zero_division=0
    )

    mcc = matthews_corrcoef(y_true, y_pred)

    bal_acc = balanced_accuracy_score(y_true, y_pred)

    fpr = fp / (fp + tn + 1e-12)
    fnr = fn / (fn + tp + 1e-12)

    # Curve-based metrics require both classes in y_true
    if len(np.unique(y_true)) == 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
    else:
        auroc = np.nan
        auprc = np.nan

    # Calibration metric
    brier = brier_score_loss(y_true, y_prob)

    return {
        "ACC"    : float(acc),
        "Sn"     : float(sn),
        "Sp"     : float(sp),
        "Pr"     : float(pr),
        "F1"     : float(f1),
        "MCC"    : float(mcc),
        "AUROC"  : float(auroc),
        "AUPRC"  : float(auprc),
        "BalACC" : float(bal_acc),
        "FPR"    : float(fpr),
        "FNR"    : float(fnr),
        "Brier"  : float(brier),
        "TP"     : int(tp),
        "TN"     : int(tn),
        "FP"     : int(fp),
        "FN"     : int(fn),
        "threshold": float(threshold)
    }

# CELL — 5-fold CV for binary classification on TRAIN split only

In [ ]:
# CELL — 5-fold CV for binary classification on TRAIN split only
#
# IMPORTANT:
#   - CV uses ONLY train_pairs / y_train_cls.
#   - test_pairs / y_test_cls are NOT used here.
#   - A fresh untrained model is created for each fold.
#   - The earlier `model = ResiduePairGNN(...)` was only for sanity check.

import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold

print("=" * 80)
print("5-fold CV on TRAIN split only")
print("=" * 80)

# ------------------------------------------------------------
# 1. Prepare TRAIN-only arrays for CV
# ------------------------------------------------------------
train_pairs_arr = np.asarray(train_pairs, dtype=np.int64)  # [B, 2]
y_train_arr     = np.asarray(y_train_cls, dtype=np.int64)  # [B]

assert train_pairs_arr.ndim == 2 and train_pairs_arr.shape[1] == 2, \
    f"train_pairs_arr should be [B, 2], got {train_pairs_arr.shape}"

assert train_pairs_arr.shape[0] == y_train_arr.shape[0], \
    "train_pairs and y_train_cls have different lengths"

assert set(np.unique(y_train_arr)).issubset({0, 1}), \
    f"Labels must be 0/1, got {np.unique(y_train_arr)}"

print(f"CV pool pairs total : {len(train_pairs_arr)}")
print(f"CV pool positives   : {int((y_train_arr == 1).sum())}")
print(f"CV pool zeros       : {int((y_train_arr == 0).sum())}")
print(f"Held-out test pairs : {len(test_pairs)}  <-- not used in CV")


# ------------------------------------------------------------
# 2. Fresh model factory
# ------------------------------------------------------------
def make_fresh_model(seed=None):
    """
    Create a fresh untrained model for one CV fold.

    We instantiate a new model inside each fold so that fold k does not
    inherit weights learned from fold k-1.
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    return ResiduePairGNN(
        node_dim=data.x.shape[1],          # 23
        edge_dim=data.edge_attr.shape[1],  # 1
        hidden_dim=96,
        heads=4,
        dropout=0.3
    ).to(device)


# ------------------------------------------------------------
# 3. Helper: pair array -> tensor [B, 2]
#    Your corrected ResiduePairGNN accepts [B, 2] or [2, B].
# ------------------------------------------------------------
def pairs_to_tensor(pair_array, device):
    arr = np.asarray(pair_array, dtype=np.int64)
    assert arr.ndim == 2 and arr.shape[1] == 2, \
        f"pair array should be [B, 2], got {arr.shape}"
    return torch.tensor(arr, dtype=torch.long, device=device)


# ------------------------------------------------------------
# 4. Train one epoch
# ------------------------------------------------------------
def train_one_epoch_pair_batches(
    model,
    data,
    pair_tensor,
    y_tensor,
    optimizer,
    criterion,
    batch_size=128
):
    """
    Train for one epoch over mini-batches of residue pairs.

    Returns:
        average training loss

    Note:
        We do NOT return training logits from this function because the
        model changes after each optimizer step. Metrics are recomputed
        after the epoch using the final model state.
    """
    model.train()

    n = pair_tensor.shape[0]
    order = torch.randperm(n, device=pair_tensor.device)

    total_loss = 0.0
    total_count = 0

    for start in range(0, n, batch_size):
        idx = order[start:start + batch_size]

        batch_pairs = pair_tensor[idx]
        batch_y     = y_tensor[idx]

        logits = model(data, batch_pairs)
        loss = criterion(logits, batch_y)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss += loss.item() * len(idx)
        total_count += len(idx)

    return total_loss / max(total_count, 1)


# ------------------------------------------------------------
# 5. Batched inference
# ------------------------------------------------------------
@torch.no_grad()
def predict_pair_batches(model, data, pair_tensor, batch_size=256):
    """
    Predict logits for all residue pairs.
    """
    model.eval()

    logits_chunks = []
    n = pair_tensor.shape[0]

    for start in range(0, n, batch_size):
        batch_pairs = pair_tensor[start:start + batch_size]
        logits = model(data, batch_pairs)
        logits_chunks.append(logits)

    return torch.cat(logits_chunks, dim=0)


# ------------------------------------------------------------
# 6. CV config
# ------------------------------------------------------------
NUM_FOLDS    = 5
MAX_EPOCHS   = 200
PATIENCE     = 15
BATCH_SIZE   = 128
LR           = 1e-3
WEIGHT_DECAY = 1e-4
THRESHOLD    = 0.5

criterion = nn.BCEWithLogitsLoss()

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=42
)

fold_results = []
best_fold_states = []
cv_histories = []


In [ ]:
# ------------------------------------------------------------
# 7. CV loop
# ------------------------------------------------------------
for fold, (tr_idx, va_idx) in enumerate(
    skf.split(train_pairs_arr, y_train_arr),
    start=1
):
    print("\n" + "=" * 80)
    print(f"Fold {fold}/{NUM_FOLDS}")
    print("=" * 80)

    fold_train_pairs = train_pairs_arr[tr_idx]
    fold_val_pairs   = train_pairs_arr[va_idx]

    y_tr = y_train_arr[tr_idx]
    y_va = y_train_arr[va_idx]

    print(
        f"Fold train size : {len(fold_train_pairs)} | "
        f"pos={int((y_tr == 1).sum())} zero={int((y_tr == 0).sum())}"
    )

    print(
        f"Fold val size   : {len(fold_val_pairs)}   | "
        f"pos={int((y_va == 1).sum())} zero={int((y_va == 0).sum())}"
    )

    pair_tr_tensor = pairs_to_tensor(fold_train_pairs, device)
    pair_va_tensor = pairs_to_tensor(fold_val_pairs, device)

    y_tr_tensor = torch.tensor(y_tr, dtype=torch.float32, device=device)
    y_va_tensor = torch.tensor(y_va, dtype=torch.float32, device=device)

    # Fresh untrained model for this fold
    fold_model = make_fresh_model(seed=42 + fold)

    optimizer = torch.optim.Adam(
        fold_model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    best_state = None
    best_epoch = -1
    best_score = -np.inf
    patience_counter = 0

    history = []

    for epoch in range(1, MAX_EPOCHS + 1):

        # -------------------- Train --------------------
        train_loss = train_one_epoch_pair_batches(
            model=fold_model,
            data=data,
            pair_tensor=pair_tr_tensor,
            y_tensor=y_tr_tensor,
            optimizer=optimizer,
            criterion=criterion,
            batch_size=BATCH_SIZE
        )

        # -------------------- Train metrics after epoch --------------------
        train_logits = predict_pair_batches(
            model=fold_model,
            data=data,
            pair_tensor=pair_tr_tensor,
            batch_size=BATCH_SIZE
        )

        train_probs = torch.sigmoid(train_logits).detach().cpu().numpy()

        train_metrics = evaluate_binary(
            y_true=y_tr,
            y_prob=train_probs,
            threshold=THRESHOLD
        )

        # -------------------- Validation --------------------
        val_logits = predict_pair_batches(
            model=fold_model,
            data=data,
            pair_tensor=pair_va_tensor,
            batch_size=BATCH_SIZE
        )

        val_loss = criterion(val_logits, y_va_tensor).item()
        val_probs = torch.sigmoid(val_logits).detach().cpu().numpy()

        val_metrics = evaluate_binary(
            y_true=y_va,
            y_prob=val_probs,
            threshold=THRESHOLD
        )

        # -------------------- Early stopping score --------------------
        early_score = (
            0.50 * val_metrics["MCC"] +
            0.30 * val_metrics["AUPRC"] +
            0.20 * val_metrics["F1"]
        )

        history.append({
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "train_metrics": copy.deepcopy(train_metrics),
            "val_metrics": copy.deepcopy(val_metrics),
            "early_score": float(early_score)
        })

        if early_score > best_score:
            best_score = early_score
            best_epoch = epoch
            best_state = copy.deepcopy(fold_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"Epoch {epoch:3d} | "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} | "
                f"val_ACC={val_metrics['ACC']:.4f} "
                f"val_Sn={val_metrics['Sn']:.4f} "
                f"val_Sp={val_metrics['Sp']:.4f} "
                f"val_Pr={val_metrics['Pr']:.4f} "
                f"val_F1={val_metrics['F1']:.4f} "
                f"val_MCC={val_metrics['MCC']:.4f} "
                f"val_AUPRC={val_metrics['AUPRC']:.4f} "
                f"val_AUROC={val_metrics['AUROC']:.4f}"
            )

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch} | best epoch = {best_epoch}")
            break

    assert best_state is not None, "No best model state was saved."

    # --------------------------------------------------------
    # Reload best fold model and recompute validation metrics
    # --------------------------------------------------------
    fold_model.load_state_dict(best_state)
    fold_model.eval()

    final_val_logits = predict_pair_batches(
        model=fold_model,
        data=data,
        pair_tensor=pair_va_tensor,
        batch_size=BATCH_SIZE
    )

    final_val_loss = criterion(final_val_logits, y_va_tensor).item()
    final_val_probs = torch.sigmoid(final_val_logits).detach().cpu().numpy()

    final_val_metrics = evaluate_binary(
        y_true=y_va,
        y_prob=final_val_probs,
        threshold=THRESHOLD
    )

    fold_summary = {
        "fold": int(fold),
        "best_epoch": int(best_epoch),
        "val_loss": float(final_val_loss),
        **final_val_metrics
    }

    fold_results.append(fold_summary)
    best_fold_states.append(copy.deepcopy(best_state))
    cv_histories.append(copy.deepcopy(history))

    print("\nBest fold summary:")
    for k, v in fold_summary.items():
        if isinstance(v, float):
            print(f"  {k:10s}: {v:.4f}")
        else:
            print(f"  {k:10s}: {v}")


# ------------------------------------------------------------
# 8. Aggregate CV results
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("Aggregate validation performance across folds")
print("=" * 80)

fold_results_df = pd.DataFrame(fold_results)

metric_names = [
    "val_loss", "ACC", "Sn", "Sp", "Pr", "F1", "MCC",
    "AUPRC", "AUROC", "BalACC", "FPR", "FNR", "Brier"
]

cv_summary = {}

for m in metric_names:
    vals = fold_results_df[m].astype(float).values
    cv_summary[m] = {
        "mean": float(vals.mean()),
        "std": float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
    }

for m in metric_names:
    print(f"{m:8s}: {cv_summary[m]['mean']:.4f} ± {cv_summary[m]['std']:.4f}")

print("\nPer-fold results:")
print(fold_results_df)


# ------------------------------------------------------------
# 9. Confirm all fold histories were saved
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("Stored fold histories")
print("=" * 80)

print(f"len(cv_histories) = {len(cv_histories)}")

for i, h in enumerate(cv_histories, start=1):
    print(f"Fold {i}: {len(h)} epochs saved")

In [ ]:
# CELL — Fold-only CV plots + cleaner metrics summary
#
# Purpose:
#   Visualize 5-fold CV results using only cv_histories and fold_results_df.
#   This does NOT use held-out test_pairs/test labels.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# 1. Safety checks
# -------------------------------------------------------------------
assert "cv_histories" in globals(), "cv_histories not found. Re-run CV with histories saved."
assert "fold_results_df" in globals(), "fold_results_df not found. Re-run CV summary cell first."

num_folds = len(cv_histories)
assert num_folds > 0, "cv_histories is empty."

assert len(fold_results_df) == num_folds, (
    f"fold_results_df has {len(fold_results_df)} rows but cv_histories has {num_folds} folds."
)

required_fold_cols = ["fold", "best_epoch", "val_loss", "ACC", "Sn", "Sp", "Pr", "F1", "MCC", "AUPRC", "AUROC", "BalACC"]
missing_cols = [c for c in required_fold_cols if c not in fold_results_df.columns]
assert not missing_cols, f"fold_results_df is missing columns: {missing_cols}"

max_epochs = max(len(h) for h in cv_histories)

print("=" * 80)
print("Fold history lengths")
print("=" * 80)
for f, hist in enumerate(cv_histories, start=1):
    print(f"Fold {f}: {len(hist)} epochs saved")


# -------------------------------------------------------------------
# 2. Convert CV histories into matrices [fold, epoch]
# -------------------------------------------------------------------
def build_history_matrix(metric_name, split=None):
    """
    Build matrix of shape [num_folds, max_epochs].

    Args:
        metric_name:
            'train_loss', 'val_loss', or a metric key such as 'ACC', 'F1', 'MCC'.
        split:
            None for loss keys;
            'train' for train_metrics;
            'val' for val_metrics.
    """
    mat = np.full((num_folds, max_epochs), np.nan, dtype=float)

    for f, hist in enumerate(cv_histories):
        for e, rec in enumerate(hist):
            if split is None:
                mat[f, e] = float(rec[metric_name])
            elif split == "train":
                mat[f, e] = float(rec["train_metrics"][metric_name])
            elif split == "val":
                mat[f, e] = float(rec["val_metrics"][metric_name])
            else:
                raise ValueError("split must be None, 'train', or 'val'.")

    return mat


train_loss_mat = build_history_matrix("train_loss", split=None)
val_loss_mat   = build_history_matrix("val_loss", split=None)

train_acc_mat  = build_history_matrix("ACC", split="train")
val_acc_mat    = build_history_matrix("ACC", split="val")

epochs = np.arange(1, max_epochs + 1)


# -------------------------------------------------------------------
# 3. Helper function for all-fold curve plot
# -------------------------------------------------------------------
def plot_cv_curves(train_mat, val_mat, ylabel, title=None, ylim=None):
    """
    Plot all folds plus mean train/val curves.
    """
    fig, ax = plt.subplots(figsize=(10, 7))

    cmap = plt.get_cmap("tab10")

    for f in range(num_folds):
        color = cmap(f % 10)

        fold_epochs = np.arange(1, np.sum(~np.isnan(train_mat[f])) + 1)

        ax.plot(
            fold_epochs,
            train_mat[f, :len(fold_epochs)],
            linestyle="-",
            linewidth=1.8,
            alpha=0.45,
            color=color,
            label=f"Train fold {f + 1}"
        )

        ax.plot(
            fold_epochs,
            val_mat[f, :len(fold_epochs)],
            linestyle="--",
            linewidth=1.8,
            alpha=0.75,
            color=color,
            label=f"Val fold {f + 1}"
        )

    # Mean curves across available folds at each epoch
    train_mean = np.nanmean(train_mat, axis=0)
    val_mean   = np.nanmean(val_mat, axis=0)

    ax.plot(
        epochs,
        train_mean,
        linestyle="-",
        linewidth=3.2,
        color="black",
        label="Train mean"
    )

    ax.plot(
        epochs,
        val_mean,
        linestyle="--",
        linewidth=3.2,
        color="black",
        label="Val mean"
    )

    ax.set_xlabel("Epoch", fontsize=28)
    ax.set_ylabel(ylabel, fontsize=28)

    if title is not None:
        ax.set_title(title, fontsize=22, pad=12)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.tick_params(axis="both", labelsize=28)
    ax.grid(True, alpha=0.3)

    # Legend can be large, but this keeps every fold visible.
    ax.legend(fontsize=16, frameon=False, ncol=2)
   # ax.set_yticks(np.linspace(ax.get_ylim()[0], ax.get_ylim()[1], 4))
    ticks = np.linspace(*ax.get_ylim(), 4)
    ax.set_yticks(ticks, labels=[f"{v:.1f}" for v in ticks])

    fig.savefig(f"{ylabel.replace(' ', '_')}.png", dpi=600, bbox_inches="tight")

    plt.tight_layout()
    plt.show()


# -------------------------------------------------------------------
# 4. Loss plot: all folds + mean
# -------------------------------------------------------------------
plot_cv_curves(
    train_mat=train_loss_mat,
    val_mat=val_loss_mat,
    ylabel="Binary Cross-Entropy Loss",
    title=None,
    ylim=None
)


# -------------------------------------------------------------------
# 5. Accuracy plot: all folds + mean
# -------------------------------------------------------------------
plot_cv_curves(
    train_mat=train_acc_mat,
    val_mat=val_acc_mat,
    ylabel="Accuracy",
    title=None,
    ylim=(0.0, 1.0)
)


# -------------------------------------------------------------------
# 6. Average validation metrics across folds
# -------------------------------------------------------------------
metric_cols = ["ACC", "Sn", "Sp", "Pr", "F1", "MCC", "AUPRC", "AUROC", "BalACC"]

avg_metrics = []

for m in metric_cols:
    vals = fold_results_df[m].astype(float).values
    avg_metrics.append({
        "Metric": m,
        "Mean": float(np.mean(vals)),
        "Std": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
        "Min": float(np.min(vals)),
        "Max": float(np.max(vals))
    })

avg_metrics_df = pd.DataFrame(avg_metrics)

print("\n" + "=" * 80)
print("Average validation metrics across folds")
print("=" * 80)

for _, row in avg_metrics_df.iterrows():
    print(
        f"{row['Metric']:8s} : "
        f"{row['Mean']:.4f} ± {row['Std']:.4f} "
        f"(range {row['Min']:.4f}–{row['Max']:.4f})"
    )


# -------------------------------------------------------------------
# 7. Metrics bar plot with individual fold points
# -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 7))

x = np.arange(len(avg_metrics_df))
means = avg_metrics_df["Mean"].values
stds  = avg_metrics_df["Std"].values
labels = avg_metrics_df["Metric"].values

ax.bar(
    x,
    means,
    yerr=stds,
    capsize=5,
    color="lightgray",
    edgecolor="black",
    linewidth=1.1,
    alpha=0.95,
    label="Mean ± SD"
)

# Overlay individual fold points
rng = np.random.default_rng(42)

for xi, metric in zip(x, labels):
    fold_vals = fold_results_df[metric].astype(float).values

    # small jitter so points do not overlap exactly
    jitter = rng.normal(loc=0.0, scale=0.035, size=len(fold_vals))

    ax.scatter(
        np.full_like(fold_vals, xi, dtype=float) + jitter,
        fold_vals,
        s=55,
        color="black",
        alpha=0.75,
        zorder=3
    )

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=22)
ax.tick_params(axis="y", labelsize=16)
ax.set_ylabel("Validation score", fontsize=22)

# Dynamic y-axis, allowing MCC below zero if needed
# y_min = min(0.0, float(np.nanmin(means - stds)) - 0.05)
# y_max = max(1.05, float(np.nanmax(means + stds)) + 0.05)

# ax.set_ylim(y_min, y_max)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False, fontsize=20)

plt.tight_layout()
plt.show()


# -------------------------------------------------------------------
# 8. Optional: compact per-fold result table
# -------------------------------------------------------------------
display_cols = ["fold", "best_epoch", "val_loss", "ACC", "Sn", "Sp", "Pr", "F1", "MCC", "AUPRC", "AUROC", "BalACC", "Brier"]

available_display_cols = [c for c in display_cols if c in fold_results_df.columns]

print("\nPer-fold validation results:")
print(fold_results_df[available_display_cols].round(4))

print("\nAverage metrics table:")
print(avg_metrics_df.round(4))

In [ ]:
# CELL — Final training on full TRAIN split and save final classifier model
#
# IMPORTANT:
#   - Uses all train_pairs / y_train_cls.
#   - Does NOT use test_pairs / y_test_cls.
#   - Uses CV-selected hyperparameters.
#   - Trains for a fixed CV-selected number of epochs.
#   - Saves the final model state, not the lowest training-loss state.

import copy
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

print("=" * 80)
print("Final training on full TRAIN split")
print("=" * 80)

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------
assert "train_pairs" in globals(), "train_pairs not found."
assert "y_train_cls" in globals(), "y_train_cls not found."
assert "fold_results_df" in globals(), "fold_results_df not found. Run CV first."
assert "cv_summary" in globals(), "cv_summary not found. Run CV summary first."
assert "fold_results" in globals(), "fold_results not found. Run CV first."

# These helpers should come from the corrected CV cell
assert "pairs_to_tensor" in globals(), "pairs_to_tensor not found. Run corrected CV cell first."
assert "train_one_epoch_pair_batches" in globals(), "train_one_epoch_pair_batches not found. Run corrected CV cell first."
assert "predict_pair_batches" in globals(), "predict_pair_batches not found. Run corrected CV cell first."
assert "evaluate_binary" in globals(), "evaluate_binary not found."

# ------------------------------------------------------------
# 2. Final model/training config
#    Use the SAME hyperparameters as the selected CV run.
# ------------------------------------------------------------
FINAL_HIDDEN_DIM = 96
FINAL_HEADS      = 4
FINAL_DROPOUT    = 0.3      # change to 0.2 only if that was your final CV-selected setting
FINAL_BATCH_SIZE = 128
FINAL_LR         = LR
FINAL_WEIGHT_DECAY = WEIGHT_DECAY
FINAL_THRESHOLD  = THRESHOLD

criterion = nn.BCEWithLogitsLoss()

print("Final training config:")
print(f"  hidden_dim   : {FINAL_HIDDEN_DIM}")
print(f"  heads        : {FINAL_HEADS}")
print(f"  dropout      : {FINAL_DROPOUT}")
print(f"  batch_size   : {FINAL_BATCH_SIZE}")
print(f"  lr           : {FINAL_LR}")
print(f"  weight_decay : {FINAL_WEIGHT_DECAY}")
print(f"  threshold    : {FINAL_THRESHOLD}")

# ------------------------------------------------------------
# 3. Prepare full training tensors
# ------------------------------------------------------------
final_train_pairs_tensor = pairs_to_tensor(train_pairs, device)
final_y_train_tensor = torch.tensor(
    y_train_cls,
    dtype=torch.float32,
    device=device
)

assert final_train_pairs_tensor.shape[0] == len(y_train_cls), (
    f"Pair/label length mismatch: {final_train_pairs_tensor.shape[0]} pairs, "
    f"{len(y_train_cls)} labels."
)

print("\nFinal train data:")
print(f"  Final train pairs : {len(train_pairs)}")
print(f"  Positives         : {int((y_train_cls == 1).sum())}")
print(f"  Zeros             : {int((y_train_cls == 0).sum())}")
print(f"  Held-out test     : {len(test_pairs)} pairs  <-- not used here")

# ------------------------------------------------------------
# 4. Choose final number of epochs from CV
# ------------------------------------------------------------
best_epochs = fold_results_df["best_epoch"].astype(int).values

mean_best_epoch   = int(round(np.mean(best_epochs)))
median_best_epoch = int(round(np.median(best_epochs)))

# Median is more robust to unstable folds.
FINAL_EPOCHS = max(1, median_best_epoch)

print("\nCV best epochs:")
print(fold_results_df[["fold", "best_epoch"]])
print(f"\nMean best epoch   : {mean_best_epoch}")
print(f"Median best epoch : {median_best_epoch}")
print(f"Final epochs used : {FINAL_EPOCHS}")

# ------------------------------------------------------------
# 5. Build fresh final model
# ------------------------------------------------------------
torch.manual_seed(42)
np.random.seed(42)

final_model = ResiduePairGNN(
    node_dim=data.x.shape[1],
    edge_dim=data.edge_attr.shape[1],
    hidden_dim=FINAL_HIDDEN_DIM,
    heads=FINAL_HEADS,
    dropout=FINAL_DROPOUT
).to(device)

optimizer = torch.optim.Adam(
    final_model.parameters(),
    lr=FINAL_LR,
    weight_decay=FINAL_WEIGHT_DECAY
)

# ------------------------------------------------------------
# 6. Final training loop
# ------------------------------------------------------------
final_history = []

for epoch in range(1, FINAL_EPOCHS + 1):

    train_loss = train_one_epoch_pair_batches(
        model=final_model,
        data=data,
        pair_tensor=final_train_pairs_tensor,
        y_tensor=final_y_train_tensor,
        optimizer=optimizer,
        criterion=criterion,
        batch_size=FINAL_BATCH_SIZE
    )

    # Recompute train metrics after the epoch using the final model state
    train_logits = predict_pair_batches(
        model=final_model,
        data=data,
        pair_tensor=final_train_pairs_tensor,
        batch_size=FINAL_BATCH_SIZE
    )

    train_probs = torch.sigmoid(train_logits).detach().cpu().numpy()

    train_metrics = evaluate_binary(
        y_true=y_train_cls,
        y_prob=train_probs,
        threshold=FINAL_THRESHOLD
    )

    final_history.append({
        "epoch": int(epoch),
        "train_loss": float(train_loss),
        "train_metrics": copy.deepcopy(train_metrics)
    })

    if epoch == 1 or epoch % 1 == 0 or epoch == FINAL_EPOCHS:
        print(
            f"Epoch {epoch:3d}/{FINAL_EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"ACC={train_metrics['ACC']:.4f} "
            f"Sn={train_metrics['Sn']:.4f} "
            f"Sp={train_metrics['Sp']:.4f} "
            f"Pr={train_metrics['Pr']:.4f} "
            f"F1={train_metrics['F1']:.4f} "
            f"MCC={train_metrics['MCC']:.4f} "
            f"AUPRC={train_metrics['AUPRC']:.4f} "
            f"AUROC={train_metrics['AUROC']:.4f}"
        )

# ------------------------------------------------------------
# 7. Save final model state
# ------------------------------------------------------------
final_model.eval()
final_state = copy.deepcopy(final_model.state_dict())

save_dir = Path("saved_models")
save_dir.mkdir(exist_ok=True)

best_model_path = save_dir / "final_binary_residue_pair_gnn.pt"

checkpoint = {
    "model_state_dict": final_state,
    "model_class": "ResiduePairGNN",

    "node_dim": int(data.x.shape[1]),
    "edge_dim": int(data.edge_attr.shape[1]),
    "hidden_dim": int(FINAL_HIDDEN_DIM),
    "heads": int(FINAL_HEADS),
    "dropout": float(FINAL_DROPOUT),

    "final_epochs": int(FINAL_EPOCHS),
    "mean_best_epoch_from_cv": int(mean_best_epoch),
    "median_best_epoch_from_cv": int(median_best_epoch),

    "lr": float(FINAL_LR),
    "weight_decay": float(FINAL_WEIGHT_DECAY),
    "batch_size": int(FINAL_BATCH_SIZE),
    "threshold": float(FINAL_THRESHOLD),

    "final_history": final_history,
    "cv_summary": cv_summary,
    "fold_results": fold_results,
    "fold_results_df": fold_results_df.to_dict(orient="records"),

    "train_pairs": train_pairs,
    "y_train_cls": y_train_cls.tolist(),
    "test_pairs": test_pairs,
    "y_test_cls": y_test_cls.tolist(),
}

torch.save(checkpoint, best_model_path)

print("\nFinal training complete.")
print(f"Saved final model checkpoint to: {best_model_path}")

In [ ]:
# CELL — Evaluate final model on held-out TEST set
#
# IMPORTANT:
#   - This is the first and only evaluation on held-out test_pairs.
#   - Do NOT tune threshold, model, dropout, or epochs after seeing this result.
#   - Use the final model trained on all train_pairs.

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("=" * 80)
print("Held-out TEST evaluation")
print("=" * 80)

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------
assert "final_model" in globals(), "final_model not found. Run final training cell first."
assert "test_pairs" in globals(), "test_pairs not found."
assert "y_test_cls" in globals(), "y_test_cls not found."
assert "pairs_to_tensor" in globals(), "pairs_to_tensor not found."
assert "evaluate_binary" in globals(), "evaluate_binary not found."

# Prefer final-training config variables if available
_eval_batch_size = FINAL_BATCH_SIZE if "FINAL_BATCH_SIZE" in globals() else BATCH_SIZE
_eval_threshold  = FINAL_THRESHOLD  if "FINAL_THRESHOLD"  in globals() else THRESHOLD

# ------------------------------------------------------------
# 2. Prepare test tensors
# ------------------------------------------------------------
test_pairs_tensor = pairs_to_tensor(test_pairs, device)

y_test_arr = np.asarray(y_test_cls, dtype=np.int64)
y_test_tensor = torch.tensor(y_test_arr, dtype=torch.float32, device=device)

assert test_pairs_tensor.shape[0] == len(y_test_arr), (
    f"Pair/label mismatch: {test_pairs_tensor.shape[0]} test pairs, "
    f"{len(y_test_arr)} test labels."
)

print(f"Test pairs : {len(test_pairs)}")
print(f"Positives  : {int((y_test_arr == 1).sum())}")
print(f"Zeros      : {int((y_test_arr == 0).sum())}")
print(f"Threshold  : {_eval_threshold:.4f}")

# ------------------------------------------------------------
# 3. Make sure final_model contains the final trained weights
# ------------------------------------------------------------
# If final_state exists from the corrected final-training cell, load it.
# Otherwise, assume final_model is already the trained model.
if "final_state" in globals():
    final_model.load_state_dict(final_state)
    print("Loaded final_state into final_model.")
else:
    print("final_state not found; using current final_model weights.")

final_model.eval()

# ------------------------------------------------------------
# 4. Predict on held-out test set
# ------------------------------------------------------------
test_logits = predict_pair_batches(
    model=final_model,
    data=data,
    pair_tensor=test_pairs_tensor,
    batch_size=_eval_batch_size
)

test_probs = torch.sigmoid(test_logits).detach().cpu().numpy()
test_pred = (test_probs >= _eval_threshold).astype(int)

assert test_probs.shape[0] == len(y_test_arr), "Prediction length mismatch."
assert np.all(np.isfinite(test_probs)), "test_probs contains NaN or inf."

# ------------------------------------------------------------
# 5. Compute and print test metrics
# ------------------------------------------------------------
test_metrics = evaluate_binary(
    y_true=y_test_arr,
    y_prob=test_probs,
    threshold=_eval_threshold
)

print("\nHeld-out TEST metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:10s}: {v:.4f}")
    else:
        print(f"{k:10s}: {v}")

# ------------------------------------------------------------
# 6. Confusion matrix
# ------------------------------------------------------------
cm = confusion_matrix(y_test_arr, test_pred, labels=[0, 1])

print("\nConfusion matrix:")
print(cm)
print("\nRows = true labels, columns = predicted labels")
print("Class 0 = true-zero pair")
print("Class 1 = nonzero-contact pair")

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Zero", "Nonzero"]
)

disp.plot(
    values_format="d",
    cmap="Blues",
    colorbar=False,
    ax=ax
)

for text in disp.text_.ravel():
    text.set_fontsize(24)

ax.set_xlabel("Predicted label", fontsize=20)
ax.set_ylabel("True label", fontsize=20)
ax.tick_params(axis="both", labelsize=16)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. Save test outputs for later plots/analysis
# ------------------------------------------------------------
test_eval_outputs = {
    "test_pairs": test_pairs,
    "y_test": y_test_arr,
    "test_logits": test_logits.detach().cpu().numpy(),
    "test_probs": test_probs,
    "test_pred": test_pred,
    "test_metrics": test_metrics,
    "confusion_matrix": cm,
    "threshold": float(_eval_threshold),
}

print("\nStored outputs in: test_eval_outputs")

In [ ]:
# CELL — Evaluate final model on held-out TEST set
#
# IMPORTANT:
#   - This is the first and only evaluation on held-out test_pairs.
#   - Do NOT tune threshold, model, dropout, or epochs after seeing this result.
#   - Use the final model trained on all train_pairs.

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("=" * 80)
print("Held-out TEST evaluation")
print("=" * 80)

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------
assert "final_model" in globals(), "final_model not found. Run final training cell first."
assert "test_pairs" in globals(), "test_pairs not found."
assert "y_test_cls" in globals(), "y_test_cls not found."
assert "pairs_to_tensor" in globals(), "pairs_to_tensor not found."
assert "evaluate_binary" in globals(), "evaluate_binary not found."

# Prefer final-training config variables if available
_eval_batch_size = FINAL_BATCH_SIZE if "FINAL_BATCH_SIZE" in globals() else BATCH_SIZE
_eval_threshold  = FINAL_THRESHOLD  if "FINAL_THRESHOLD"  in globals() else THRESHOLD

# ------------------------------------------------------------
# 2. Prepare test tensors
# ------------------------------------------------------------
test_pairs_tensor = pairs_to_tensor(test_pairs, device)

y_test_arr = np.asarray(y_test_cls, dtype=np.int64)
y_test_tensor = torch.tensor(y_test_arr, dtype=torch.float32, device=device)

assert test_pairs_tensor.shape[0] == len(y_test_arr), (
    f"Pair/label mismatch: {test_pairs_tensor.shape[0]} test pairs, "
    f"{len(y_test_arr)} test labels."
)

print(f"Test pairs : {len(test_pairs)}")
print(f"Positives  : {int((y_test_arr == 1).sum())}")
print(f"Zeros      : {int((y_test_arr == 0).sum())}")
print(f"Threshold  : {_eval_threshold:.4f}")

# ------------------------------------------------------------
# 3. Make sure final_model contains the final trained weights
# ------------------------------------------------------------
# If final_state exists from the corrected final-training cell, load it.
# Otherwise, assume final_model is already the trained model.
if "final_state" in globals():
    final_model.load_state_dict(final_state)
    print("Loaded final_state into final_model.")
else:
    print("final_state not found; using current final_model weights.")

final_model.eval()

# ------------------------------------------------------------
# 4. Predict on held-out test set
# ------------------------------------------------------------
test_logits = predict_pair_batches(
    model=final_model,
    data=data,
    pair_tensor=test_pairs_tensor,
    batch_size=_eval_batch_size
)

test_probs = torch.sigmoid(test_logits).detach().cpu().numpy()
test_pred = (test_probs >= _eval_threshold).astype(int)

assert test_probs.shape[0] == len(y_test_arr), "Prediction length mismatch."
assert np.all(np.isfinite(test_probs)), "test_probs contains NaN or inf."

# ------------------------------------------------------------
# 5. Compute and print test metrics
# ------------------------------------------------------------
test_metrics = evaluate_binary(
    y_true=y_test_arr,
    y_prob=test_probs,
    threshold=_eval_threshold
)

print("\nHeld-out TEST metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:10s}: {v:.4f}")
    else:
        print(f"{k:10s}: {v}")

# ------------------------------------------------------------
# ------------------------------------------------------------
# 6. Row-normalized confusion matrix (%)
# ------------------------------------------------------------

# Raw confusion-matrix counts
cm = confusion_matrix(
    y_test_arr,
    test_pred,
    labels=[0, 1]
)

# Normalize each true-label row to 100%
cm_percent = confusion_matrix(
    y_test_arr,
    test_pred,
    labels=[0, 1],
    normalize="true"
) * 100.0

print("\nRaw confusion-matrix counts:")
print(cm)

print("\nRow-normalized confusion matrix (%):")
print(np.round(cm_percent, 2))

print("\nRows = true labels, columns = predicted labels")
print("Class 0 = true-zero pair")
print("Class 1 = nonzero-contact pair")

fig, ax = plt.subplots(figsize=(6, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_percent,
    display_labels=["Zero", "Nonzero"]
)

disp.plot(
    values_format=".1f",
    cmap="Blues",
    colorbar=False,
    ax=ax
)

# Add percentage signs and adjust annotation font
for text in disp.text_.ravel():
    text.set_text(f"{float(text.get_text()):.1f}%")
    text.set_fontsize(24)
    text.set_fontweight("bold")

ax.set_xlabel("Predicted label", fontsize=20)
ax.set_ylabel("True label", fontsize=20)

ax.tick_params(
    axis="both",
    labelsize=16
)

plt.tight_layout()
plt.show()

test_eval_outputs = {
    "test_pairs": test_pairs,
    "y_test": y_test_arr,
    "test_logits": test_logits.detach().cpu().numpy(),
    "test_probs": test_probs,
    "test_pred": test_pred,
    "test_metrics": test_metrics,
    "confusion_matrix": cm,
    "confusion_matrix_percent": cm_percent,
    "threshold": float(_eval_threshold),
}


In [ ]:
# CELL — Build full 103×103 interaction matrices using MD-known values + GNN binary predictions
#
# IMPORTANT CONCEPT:
#   The GNN is a binary classifier.
#   sigmoid(logit) = P(pair is nonzero/contact)
#   It is NOT the MD contact probability magnitude.
#
# Therefore:
#   - Do NOT treat sigmoid output as normalized MD contact probability.
#   - Multiplying sigmoid output by p_max creates only a raw-scale heuristic score.
#
# This cell saves:
#   1. full_class_probability_matrix:
#        MD positive = 1, MD zero = 0, unknown = GNN P(nonzero)
#   2. full_binary_interaction_matrix:
#        MD positive = 1, MD zero = 0, unknown = GNN 0/1 class
#   3. full_md_raw_or_rawlike_matrix:
#        MD known = true raw MD probability from all_edges
#        unknown = GNN P(nonzero) * p_max_cls as a heuristic raw-like score
#   4. source_matrix:
#        where each value came from

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix
from matplotlib.ticker import MaxNLocator

print("=" * 80)
print("Building full 103 x 103 residue interaction matrices")
print("=" * 80)

# ------------------------------------------------------------
# 0. Safety checks
# ------------------------------------------------------------
assert "final_model" in globals(), "final_model not found. Run final training cell first."
assert "all_edges" in globals(), "all_edges not found."
assert "all_possible_pairs" in globals(), "all_possible_pairs not found."
assert "positive_pairs" in globals(), "positive_pairs not found."
assert "zero_pairs" in globals(), "zero_pairs not found."
assert "unknown_pairs" in globals(), "unknown_pairs not found."
assert "pairs_to_tensor" in globals(), "pairs_to_tensor not found."
assert "predict_pair_batches" in globals(), "predict_pair_batches not found."

# Prefer final-training settings if available
_eval_batch_size = FINAL_BATCH_SIZE if "FINAL_BATCH_SIZE" in globals() else BATCH_SIZE
_eval_threshold  = FINAL_THRESHOLD  if "FINAL_THRESHOLD"  in globals() else THRESHOLD

# For raw-like heuristic scaling of GNN probabilities
# Prefer p_max_cls because it belongs to the binary split.
if "p_max_cls" in globals():
    raw_like_scale = float(p_max_cls)
elif "target_scale" in globals():
    raw_like_scale = float(target_scale)
elif "p_max" in globals():
    raw_like_scale = float(p_max)
else:
    raw_like_scale = float(max(all_edges.values()))

print(f"Threshold used for binary unknown calls : {_eval_threshold:.4f}")
print(f"Raw-like scale for GNN heuristic        : {raw_like_scale:.12e}")

# ------------------------------------------------------------
# 1. Define MD-known and unknown pair sets
# ------------------------------------------------------------
md_known_pairs = set(positive_pairs) | set(zero_pairs)
missing_pairs = sorted(set(unknown_pairs))

assert md_known_pairs == (set(all_possible_pairs) - set(missing_pairs)), \
    "MD-known + missing pair partition is inconsistent."

assert len(md_known_pairs & set(missing_pairs)) == 0, \
    "Overlap found between MD-known and missing pairs."

print(f"Total possible residue pairs        : {len(all_possible_pairs)}")
print(f"MD-known pairs                      : {len(md_known_pairs)}")
print(f"  MD nonzero positive pairs         : {len(positive_pairs)}")
print(f"  MD true-zero pairs                : {len(zero_pairs)}")
print(f"Unknown pairs predicted by GNN      : {len(missing_pairs)}")

# ------------------------------------------------------------
# 2. Initialize full matrices
# ------------------------------------------------------------
full_class_probability_matrix = np.zeros((N, N), dtype=float)
full_binary_interaction_matrix = np.zeros((N, N), dtype=float)

# Known MD raw matrix:
#   Unknown pairs are NaN here because there is no true MD magnitude.
full_md_raw_known_matrix = np.full((N, N), np.nan, dtype=float)

# Hybrid raw-like score:
#   MD known = true raw MD value
#   Unknown = GNN probability * raw_like_scale
# This is a heuristic score, not true MD probability.
full_md_raw_or_rawlike_matrix = np.zeros((N, N), dtype=float)

source_matrix = np.full((N, N), "diag", dtype=object)

# Diagonal
np.fill_diagonal(full_class_probability_matrix, 0.0)
np.fill_diagonal(full_binary_interaction_matrix, 0.0)
np.fill_diagonal(full_md_raw_known_matrix, 0.0)
np.fill_diagonal(full_md_raw_or_rawlike_matrix, 0.0)

# ------------------------------------------------------------
# 3. Fill MD-known positive pairs using all_edges
#    all_edges is already undirected and merged correctly.
# ------------------------------------------------------------
for pair in positive_pairs:
    i, j = pair

    md_raw_val = float(all_edges[pair])

    # Binary/class-probability interpretation
    full_class_probability_matrix[i, j] = 1.0
    full_class_probability_matrix[j, i] = 1.0

    full_binary_interaction_matrix[i, j] = 1.0
    full_binary_interaction_matrix[j, i] = 1.0

    # True MD raw value
    full_md_raw_known_matrix[i, j] = md_raw_val
    full_md_raw_known_matrix[j, i] = md_raw_val

    full_md_raw_or_rawlike_matrix[i, j] = md_raw_val
    full_md_raw_or_rawlike_matrix[j, i] = md_raw_val

    source_matrix[i, j] = "MD_nonzero"
    source_matrix[j, i] = "MD_nonzero"

# ------------------------------------------------------------
# 4. Fill MD-known true-zero pairs
# ------------------------------------------------------------
for pair in zero_pairs:
    i, j = pair

    full_class_probability_matrix[i, j] = 0.0
    full_class_probability_matrix[j, i] = 0.0

    full_binary_interaction_matrix[i, j] = 0.0
    full_binary_interaction_matrix[j, i] = 0.0

    full_md_raw_known_matrix[i, j] = 0.0
    full_md_raw_known_matrix[j, i] = 0.0

    full_md_raw_or_rawlike_matrix[i, j] = 0.0
    full_md_raw_or_rawlike_matrix[j, i] = 0.0

    source_matrix[i, j] = "MD_zero"
    source_matrix[j, i] = "MD_zero"

# ------------------------------------------------------------
# 5. Predict unknown pairs with final binary GNN
# ------------------------------------------------------------
if "final_state" in globals():
    final_model.load_state_dict(final_state)
    print("Loaded final_state into final_model.")
else:
    print("final_state not found; using current final_model weights.")

final_model.eval()

missing_pairs_tensor = pairs_to_tensor(missing_pairs, device)

missing_logits = predict_pair_batches(
    model=final_model,
    data=data,
    pair_tensor=missing_pairs_tensor,
    batch_size=_eval_batch_size
)

missing_probs = torch.sigmoid(missing_logits).detach().cpu().numpy()
missing_pred = (missing_probs >= _eval_threshold).astype(int)

assert len(missing_probs) == len(missing_pairs)

# Fill unknown predictions
for pair, prob, pred in zip(missing_pairs, missing_probs, missing_pred):
    i, j = pair

    prob = float(prob)
    pred = int(pred)

    # Probability that pair is nonzero/contact
    full_class_probability_matrix[i, j] = prob
    full_class_probability_matrix[j, i] = prob

    # Binary predicted interaction
    full_binary_interaction_matrix[i, j] = pred
    full_binary_interaction_matrix[j, i] = pred

    # Unknown has no true MD raw value
    full_md_raw_known_matrix[i, j] = np.nan
    full_md_raw_known_matrix[j, i] = np.nan

    # Heuristic raw-like score
    # This is NOT true MD probability magnitude.
    rawlike_val = prob * raw_like_scale
    full_md_raw_or_rawlike_matrix[i, j] = rawlike_val
    full_md_raw_or_rawlike_matrix[j, i] = rawlike_val

    src = "GNN_predicted_nonzero" if pred == 1 else "GNN_predicted_zero"
    source_matrix[i, j] = src
    source_matrix[j, i] = src

# ------------------------------------------------------------
# 6. Matrix sanity checks
# ------------------------------------------------------------
for name, mat in [
    ("class_probability", full_class_probability_matrix),
    ("binary_interaction", full_binary_interaction_matrix),
    ("md_raw_or_rawlike", full_md_raw_or_rawlike_matrix),
]:
    assert np.allclose(mat, mat.T, equal_nan=True), f"{name} matrix is not symmetric."

print("\nUnknown GNN prediction summary:")
print(f"Unknown pairs total              : {len(missing_pairs)}")
print(f"GNN predicted nonzero pairs      : {int((missing_pred == 1).sum())}")
print(f"GNN predicted zero pairs         : {int((missing_pred == 0).sum())}")
print(f"Mean GNN P(nonzero), unknown     : {missing_probs.mean():.4f}")
print(f"Median GNN P(nonzero), unknown   : {np.median(missing_probs):.4f}")
print(f"Min/Max GNN P(nonzero), unknown  : {missing_probs.min():.4f} / {missing_probs.max():.4f}")

print("\nFull class-probability matrix summary:")
print(f"Shape             : {full_class_probability_matrix.shape}")
print(f"Min/Max           : {full_class_probability_matrix.min():.4f} / {full_class_probability_matrix.max():.4f}")
print(f"Nonzero entries   : {np.count_nonzero(full_class_probability_matrix)}")

print("\nFull binary-interaction matrix summary:")
print(f"Nonzero entries   : {np.count_nonzero(full_binary_interaction_matrix)}")

print("\nFull raw/rawl ike matrix summary:")
print(f"Raw-like scale used            : {raw_like_scale:.12e}")
print(f"Min/Max raw-like values        : {np.nanmin(full_md_raw_or_rawlike_matrix):.12e} / {np.nanmax(full_md_raw_or_rawlike_matrix):.12e}")
print(f"Nonzero entries                : {np.count_nonzero(full_md_raw_or_rawlike_matrix)}")

# ------------------------------------------------------------
# 7. Build residue labels
# ------------------------------------------------------------
residue_letters = [global_to_info[g]["aa"] for g in range(N)]
residue_labels = [f"{global_to_info[g]['aa']}{g+1}" for g in range(N)]

# ------------------------------------------------------------
# 8. Save full matrices
# ------------------------------------------------------------
out_dir = Path("gnn_full_interaction_outputs")
out_dir.mkdir(exist_ok=True)

class_prob_df = pd.DataFrame(
    full_class_probability_matrix,
    index=residue_labels,
    columns=residue_labels
)

binary_df = pd.DataFrame(
    full_binary_interaction_matrix,
    index=residue_labels,
    columns=residue_labels
)

md_raw_known_df = pd.DataFrame(
    full_md_raw_known_matrix,
    index=residue_labels,
    columns=residue_labels
)

rawlike_df = pd.DataFrame(
    full_md_raw_or_rawlike_matrix,
    index=residue_labels,
    columns=residue_labels
)

source_matrix_df = pd.DataFrame(
    source_matrix,
    index=residue_labels,
    columns=residue_labels
)

class_prob_df.to_csv(out_dir / "full_103x103_class_probability_matrix.csv")
binary_df.to_csv(out_dir / "full_103x103_binary_interaction_matrix.csv")
md_raw_known_df.to_csv(out_dir / "full_103x103_md_raw_known_matrix_unknown_nan.csv")
rawlike_df.to_csv(out_dir / "full_103x103_md_raw_or_rawlike_heuristic_matrix.csv")
source_matrix_df.to_csv(out_dir / "full_103x103_interaction_source_matrix.csv")

print("\nSaved matrices:")
print(out_dir / "full_103x103_class_probability_matrix.csv")
print(out_dir / "full_103x103_binary_interaction_matrix.csv")
print(out_dir / "full_103x103_md_raw_known_matrix_unknown_nan.csv")
print(out_dir / "full_103x103_md_raw_or_rawlike_heuristic_matrix.csv")
print(out_dir / "full_103x103_interaction_source_matrix.csv")

# ------------------------------------------------------------
# 9. Residue-wise summaries
# ------------------------------------------------------------
# Cleanest binary-classifier residue score:
#   Sum of P(nonzero) across all other residues.
residue_total_class_probability = full_class_probability_matrix.sum(axis=1)

# Binary count of interacting pairs
residue_total_binary_count = full_binary_interaction_matrix.sum(axis=1)

# True MD-known raw sum only, ignoring unknown NaNs
residue_total_md_known_raw = np.nansum(full_md_raw_known_matrix, axis=1)

# Hybrid raw-like heuristic sum
residue_total_rawlike = full_md_raw_or_rawlike_matrix.sum(axis=1)

residue_summary_df = pd.DataFrame({
    "global_index_0based": np.arange(N),
    "residue_number_1based": np.arange(1, N + 1),
    "residue_name": residue_letters,
    "residue_label": residue_labels,

    "total_class_probability_sum": residue_total_class_probability,
    "total_binary_interaction_count": residue_total_binary_count,
    "total_md_known_raw_probability_sum": residue_total_md_known_raw,
    "total_rawlike_heuristic_sum": residue_total_rawlike,
})

residue_summary_df.to_csv(
    out_dir / "residue_total_interaction_scores.csv",
    index=False
)

print("\nSaved residue summary:")
print(out_dir / "residue_total_interaction_scores.csv")

print("\nFirst 10 residue totals:")
print(residue_summary_df.head(10))

# ------------------------------------------------------------
# 10. Gaussian smoothing without scipy
# ------------------------------------------------------------
def gaussian_smooth_1d(y, sigma=2.0, radius=6):
    y = np.asarray(y, dtype=float)
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-(x ** 2) / (2 * sigma ** 2))
    kernel = kernel / kernel.sum()
    return np.convolve(y, kernel, mode="same")

sigma = 2.0
radius = 6

residue_total_class_probability_smooth = gaussian_smooth_1d(
    residue_total_class_probability,
    sigma=sigma,
    radius=radius
)

residue_total_rawlike_smooth = gaussian_smooth_1d(
    residue_total_rawlike,
    sigma=sigma,
    radius=radius
)


################## SUM smoothing ##################

# def moving_average_smooth_1d(y, window=5):
#     """
#     Smooth a 1D residue-wise curve using a centered moving average.

#     window=5 means each residue is replaced by the average of:
#         residue g-2, g-1, g, g+1, g+2

#     Edge-safe padding is used so the first/last residues are not artificially
#     pulled downward by zero-padding.
#     """
#     y = np.asarray(y, dtype=float)

#     assert window >= 1, "window must be >= 1"
#     assert window % 2 == 1, "Use an odd window size, e.g., 3, 5, 7"

#     radius = window // 2
#     kernel = np.ones(window, dtype=float) / window

#     # Edge-safe padding: repeat boundary values
#     y_pad = np.pad(y, pad_width=radius, mode="edge")

#     return np.convolve(y_pad, kernel, mode="valid")


# window = 13

# residue_total_class_probability_smooth = moving_average_smooth_1d(
#     residue_total_class_probability,
#     window=window
# )

# residue_total_rawlike_smooth = moving_average_smooth_1d(
#     residue_total_rawlike,
#     window=window
# )
###################

residue_summary_df["total_class_probability_sum_smoothed"] = residue_total_class_probability_smooth
residue_summary_df["total_rawlike_heuristic_sum_smoothed"] = residue_total_rawlike_smooth

residue_summary_df.to_csv(
    out_dir / "residue_total_interaction_scores_with_smoothing.csv",
    index=False
)

# ------------------------------------------------------------
# 11. Plot residue-wise interaction tendency
# ------------------------------------------------------------
xpos = np.arange(N)

plt.figure(figsize=(22, 7))

plt.plot(
    xpos,
    residue_total_class_probability,
    linewidth=1.8,
    alpha=0.45,
    label="Residue total: sum of P(nonzero)"
)

plt.plot(
    xpos,
    residue_total_class_probability_smooth,
    linewidth=3.0,
    label=f"Smoothed sum of P(nonzero), sigma={sigma}"
)



plt.ylabel("Residue-wise interaction tendency", fontsize=25)

# plt.xticks(
#     xpos,
#     residue_letters,
#     rotation=90,
#     fontsize=10
# )

plt.xticks([])


plt.yticks(fontsize=18)
plt.grid(axis="y", alpha=0.3)
plt.legend(fontsize=14, frameon=False)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(4.5)

plt.tight_layout()

plt.savefig(out_dir / "residue_total_class_probability_plot.png", dpi=600)
plt.show()

print("\nSaved plot:")
print(out_dir / "residue_total_class_probability_plot.png")

# ------------------------------------------------------------
# 12. Optional plot: raw-like heuristic score
# ------------------------------------------------------------
plt.figure(figsize=(28, 6))

plt.plot(
    xpos,
    residue_total_rawlike,
    color="#79AEE3",
    linewidth=1.8,
    alpha=0.55,
    zorder=2
)

plt.plot(
    xpos,
    residue_total_rawlike_smooth,
    color="#E87500",
    linewidth=5.5,
    alpha=1.0,
    zorder=4
)




#plt.ylabel("Total interaction probability", fontsize=32)

plt.xticks([])

plt.xlim(-1,103)

#plt.yticks(fontsize=30)


ax = plt.gca()
ax.yaxis.set_major_locator(MaxNLocator(nbins=4))  # Usually gives 4 y-axis values
ax.tick_params(axis="y", labelsize=30)

plt.grid(axis="y", alpha=0.3)
#plt.legend(fontsize=14, frameon=False)

ax = plt.gca()

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(4.5)

plt.tight_layout()

plt.savefig(out_dir / "residue_total_rawlike_heuristic_plot.png", dpi=600)
plt.show()

print("\nSaved plot:")
print(out_dir / "residue_total_rawlike_heuristic_plot.png")

In [ ]:
# ============================================================
# CELL — Plot continuous 2D heatmap of full adjacency matrix
# MD-known raw values + GNN raw-like heuristic values
#
# IMPORTANT:
#   MD-known pairs = true raw MD contact probabilities from all_edges
#   Unknown pairs  = GNN P(nonzero) × raw_like_scale
#
# Therefore, this is a "raw-like heuristic" heatmap, not a true
# MD contact-probability heatmap for the GNN-predicted region.
#
# White = zero, red = higher interaction score
# Residues ordered left→right and top→bottom as 1→103
# ============================================================
 

print("=" * 80)
print("Plotting continuous full adjacency heatmap")
print("=" * 80)

out_dir = Path("gnn_full_interaction_outputs")
out_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------
# 1. Load the correct raw-like matrix from the previous corrected cell
# ------------------------------------------------------------
if "full_md_raw_or_rawlike_matrix" in globals():
    adjacency_matrix = full_md_raw_or_rawlike_matrix.copy()
    matrix_source = "memory: full_md_raw_or_rawlike_matrix"

else:
    matrix_path = out_dir / "full_103x103_md_raw_or_rawlike_heuristic_matrix.csv"

    if not matrix_path.exists():
        raise FileNotFoundError(
            "Could not find full_md_raw_or_rawlike_matrix in memory or saved CSV file. "
            "Run the corrected full-matrix construction cell first."
        )

    adjacency_matrix = pd.read_csv(matrix_path, index_col=0).values
    matrix_source = str(matrix_path)

adjacency_matrix = np.asarray(adjacency_matrix, dtype=float)

assert adjacency_matrix.shape == (N, N), (
    f"Expected adjacency matrix shape {(N, N)}, got {adjacency_matrix.shape}"
)

assert np.allclose(adjacency_matrix, adjacency_matrix.T, equal_nan=True), \
    "Adjacency matrix is not symmetric."

np.fill_diagonal(adjacency_matrix, 0.0)

print(f"Matrix source : {matrix_source}")
print(f"Matrix shape  : {adjacency_matrix.shape}")
print(f"Min / Max     : {np.nanmin(adjacency_matrix):.12e} / {np.nanmax(adjacency_matrix):.12e}")
print(f"Nonzero count : {np.count_nonzero(adjacency_matrix)}")

# ------------------------------------------------------------
# 2. Force residue order from 1→103
# ------------------------------------------------------------
order = np.arange(N)
adjacency_matrix = adjacency_matrix[order][:, order]

ordered_residue_labels = [residue_labels[i] for i in order]
ordered_residue_letters = [residue_letters[i] for i in order]

# ------------------------------------------------------------
# 3. Save ordered matrix with residue labels
# ------------------------------------------------------------
adjacency_df = pd.DataFrame(
    adjacency_matrix,
    index=ordered_residue_labels,
    columns=ordered_residue_labels,
)

ordered_matrix_path = out_dir / "continuous_full_adjacency_rawlike_MD_plus_GNN_ordered.csv"
adjacency_df.to_csv(ordered_matrix_path)

# ------------------------------------------------------------
# 4. Choose heatmap scaling
# ------------------------------------------------------------
nonzero_vals = adjacency_matrix[adjacency_matrix > 0]

if len(nonzero_vals) == 0:
    raise ValueError("Adjacency matrix has no nonzero values to plot.")

# Use full max by default.
# For very skewed matrices, percentile scaling makes weak interactions visible.
vmax_full = float(np.nanmax(adjacency_matrix))
vmax_robust = float(np.nanpercentile(nonzero_vals, 99))

USE_ROBUST_VMAX = False

vmax = vmax_robust if USE_ROBUST_VMAX else vmax_full

print(f"vmax full   : {vmax_full:.12e}")
print(f"vmax 99 pct : {vmax_robust:.12e}")
print(f"vmax used   : {vmax:.12e}")

# ------------------------------------------------------------
# 5. Plot heatmap
# origin='upper': residue 1 starts at top of y-axis
# x-axis: residue 1→103 left to right
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 14))

im = ax.imshow(
    adjacency_matrix,
    origin="lower",
    aspect="equal",
    interpolation="nearest",
    cmap="Reds",
    vmin=0.0,
    vmax=vmax,
)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(
    "Interaction probability",
    fontsize=25,
)
cbar.ax.tick_params(labelsize=16)

ax.set_xticks(np.arange(N))
ax.set_yticks(np.arange(N))

ax.set_xticklabels(
    ordered_residue_letters,
    rotation=90,
    fontsize=10
)

ax.set_yticklabels(
    ordered_residue_letters,
    fontsize=10
)

# ax.set_xlabel("Residue position 1→103", fontsize=18)
# ax.set_ylabel("Residue position 1→103", fontsize=18)

# Optional: add light grid every 8 residues to show fragment stride boundaries
for boundary in range(8, N, 8):
    ax.axhline(boundary - 0.5, color="black", linewidth=0.25, alpha=0.18)
    ax.axvline(boundary - 0.5, color="black", linewidth=0.25, alpha=0.18)

plt.tight_layout()

heatmap_path = out_dir / "continuous_full_adjacency_heatmap_rawlike_MD_plus_GNN_ordered.png"
plt.savefig(
    heatmap_path,
    dpi=600,
    bbox_inches="tight",
)

plt.show()

print("\nSaved:")
print(ordered_matrix_path)
print(heatmap_path)

In [ ]:
# CELL — Inspect GNN probabilities for unknown cross-fragment pairs
#
# IMPORTANT:
#   These are NOT "MD-zero" pairs.
#   They are unknown / unobserved pairs not covered by isolated fragment MD.
#
#   gnn_probability = P(pair is nonzero/contact)
#   It is NOT the MD contact-probability magnitude.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 70)
print("Unknown-pair GNN probability diagnostics")
print("=" * 70)

# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------
assert "missing_pairs" in globals(), "missing_pairs not found. Run full-matrix prediction cell first."
assert "missing_probs" in globals(), "missing_probs not found. Run full-matrix prediction cell first."

missing_pairs_arr = np.asarray(missing_pairs, dtype=np.int64)
missing_probs_arr = np.asarray(missing_probs, dtype=float).reshape(-1)

assert missing_pairs_arr.ndim == 2 and missing_pairs_arr.shape[1] == 2, \
    f"missing_pairs should be [B, 2], got {missing_pairs_arr.shape}"

assert len(missing_pairs_arr) == len(missing_probs_arr), \
    f"Length mismatch: {len(missing_pairs_arr)} missing pairs but {len(missing_probs_arr)} probabilities."

assert np.all(np.isfinite(missing_probs_arr)), "missing_probs contains NaN or inf."

assert np.all((missing_probs_arr >= 0.0) & (missing_probs_arr <= 1.0)), \
    "missing_probs should be probabilities in [0, 1]."

# Use final threshold if available
_diag_threshold = FINAL_THRESHOLD if "FINAL_THRESHOLD" in globals() else THRESHOLD

# ------------------------------------------------------------
# 2. Build diagnostic dataframe
# ------------------------------------------------------------
unknown_diag_df = pd.DataFrame({
    "pair_i_0based": missing_pairs_arr[:, 0],
    "pair_j_0based": missing_pairs_arr[:, 1],
    "pair_i_1based": missing_pairs_arr[:, 0] + 1,
    "pair_j_1based": missing_pairs_arr[:, 1] + 1,
    "res_i": [global_to_info[int(i)]["aa"] for i in missing_pairs_arr[:, 0]],
    "res_j": [global_to_info[int(j)]["aa"] for j in missing_pairs_arr[:, 1]],
    "sequence_separation": np.abs(missing_pairs_arr[:, 0] - missing_pairs_arr[:, 1]),
    "gnn_probability_nonzero": missing_probs_arr,
})

unknown_diag_df["predicted_label"] = (
    unknown_diag_df["gnn_probability_nonzero"] >= _diag_threshold
).astype(int)

unknown_diag_df["pair_label"] = (
    unknown_diag_df["res_i"] + unknown_diag_df["pair_i_1based"].astype(str)
    + "-"
    + unknown_diag_df["res_j"] + unknown_diag_df["pair_j_1based"].astype(str)
)

print("\nGNN probability summary for unknown pairs:")
print(unknown_diag_df["gnn_probability_nonzero"].describe())

print("\nThreshold diagnostics:")
for th in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    n_pos = int((unknown_diag_df["gnn_probability_nonzero"] >= th).sum())
    print(
        f"Threshold {th:.2f}: "
        f"{n_pos} / {len(unknown_diag_df)} unknown pairs predicted positive "
        f"({100 * n_pos / len(unknown_diag_df):.1f}%)"
    )

print(f"\nCurrent model threshold: {_diag_threshold:.3f}")
print(
    f"Predicted positive at current threshold: "
    f"{int(unknown_diag_df['predicted_label'].sum())} / {len(unknown_diag_df)} "
    f"({100 * unknown_diag_df['predicted_label'].mean():.1f}%)"
)

# ------------------------------------------------------------
# 3. Sequence-separation binned summary
# ------------------------------------------------------------
bins = [0, 2, 4, 8, 12, 16, 32, 64, 103]
labels = [
    "1-2",
    "3-4",
    "5-8",
    "9-12",
    "13-16",
    "17-32",
    "33-64",
    "65-103",
]

unknown_diag_df["separation_bin"] = pd.cut(
    unknown_diag_df["sequence_separation"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True,
)

sep_summary_df = (
    unknown_diag_df
    .groupby("separation_bin", observed=False)
    .agg(
        n_pairs=("gnn_probability_nonzero", "size"),
        mean_prob=("gnn_probability_nonzero", "mean"),
        median_prob=("gnn_probability_nonzero", "median"),
        min_prob=("gnn_probability_nonzero", "min"),
        max_prob=("gnn_probability_nonzero", "max"),
        predicted_positive_fraction=("predicted_label", "mean"),
    )
    .reset_index()
)

print("\nProbability summary by sequence-separation bin:")
print(sep_summary_df)

# ------------------------------------------------------------
# 4. Show top predicted unknown pairs
# ------------------------------------------------------------
top_k = 20

top_unknown_df = (
    unknown_diag_df
    .sort_values("gnn_probability_nonzero", ascending=False)
    .head(top_k)
    .copy()
)

print(f"\nTop {top_k} unknown pairs by GNN P(nonzero):")
print(
    top_unknown_df[
        [
            "pair_label",
            "sequence_separation",
            "gnn_probability_nonzero",
            "predicted_label",
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 5. Save diagnostics
# ------------------------------------------------------------
out_dir = Path("gnn_full_interaction_outputs")
out_dir.mkdir(exist_ok=True)

unknown_diag_df.to_csv(
    out_dir / "unknown_pair_gnn_probability_diagnostics.csv",
    index=False
)

sep_summary_df.to_csv(
    out_dir / "unknown_pair_probability_by_sequence_separation.csv",
    index=False
)

top_unknown_df.to_csv(
    out_dir / "top_unknown_pairs_by_gnn_probability.csv",
    index=False
)

print("\nSaved:")
print(out_dir / "unknown_pair_gnn_probability_diagnostics.csv")
print(out_dir / "unknown_pair_probability_by_sequence_separation.csv")
print(out_dir / "top_unknown_pairs_by_gnn_probability.csv")

# ------------------------------------------------------------
# 6. Histogram of unknown-pair probabilities
# ------------------------------------------------------------
plt.figure(figsize=(7, 5))

plt.hist(
    unknown_diag_df["gnn_probability_nonzero"],
    bins=40,
    edgecolor="black",
    alpha=0.85,
)

plt.axvline(
    _diag_threshold,
    linestyle="--",
    linewidth=2.0,
    label=f"Threshold = {_diag_threshold:.2f}",
)

plt.xlabel("GNN probability of nonzero interaction", fontsize=14)
plt.ylabel("Number of unknown pairs", fontsize=14)
plt.title("Distribution of GNN predictions for unknown pairs", fontsize=14)
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()

plt.savefig(
    out_dir / "unknown_pair_gnn_probability_histogram.png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()

# ------------------------------------------------------------
# 7. Probability vs sequence separation
# ------------------------------------------------------------
plt.figure(figsize=(7, 5))

plt.scatter(
    unknown_diag_df["sequence_separation"],
    unknown_diag_df["gnn_probability_nonzero"],
    s=8,
    alpha=0.4,
)

plt.axhline(
    _diag_threshold,
    linestyle="--",
    linewidth=2.0,
    label=f"Threshold = {_diag_threshold:.2f}",
)

plt.xlabel("Sequence separation |i - j|", fontsize=14)
plt.ylabel("GNN probability of nonzero interaction", fontsize=14)
plt.title("Unknown-pair predictions vs sequence separation", fontsize=14)
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()

plt.savefig(
    out_dir / "unknown_pair_probability_vs_sequence_separation.png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()

print("\nSaved plots:")
print(out_dir / "unknown_pair_gnn_probability_histogram.png")
print(out_dir / "unknown_pair_probability_vs_sequence_separation.png")

In [ ]:
# ============================================================================
# PUBLICATION-QUALITY 5-FOLD ROC AND PRECISION-RECALL CURVES
#
# Run this AFTER the 5-fold CV training cell.
#
# Required variables:
#   best_fold_states
#   train_pairs
#   y_train_cls
#   data
#   device
#   ResiduePairGNN
#
# Outputs:
#   cv_curve_outputs/Figure_5fold_ROC_PR_curves.png/pdf
#   cv_curve_outputs/Figure_5fold_ROC_only.png/pdf
#   cv_curve_outputs/Figure_5fold_PR_only.png/pdf
#   cv_curve_outputs/cv_fold_roc_pr_curve_points.csv
#   cv_curve_outputs/cv_fold_curve_metrics.csv
# ============================================================================

import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)

# ----------------------------------------------------------------------------
# 0. Settings
# ----------------------------------------------------------------------------
CV_RANDOM_STATE = 42
N_SPLITS = 5

# Must match the CV model architecture used when best_fold_states were saved.
CV_HIDDEN_DIM = 96
CV_HEADS = 4
CV_DROPOUT = 0.3       # change to 0.2 only if that was the CV setting you actually used

CURVE_BATCH_SIZE = 1024

CURVE_DIR = Path("cv_curve_outputs")
CURVE_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 11,
    "axes.labelsize": 22,
    "axes.titlesize": 22,
    "axes.titleweight": "bold",
    "axes.linewidth": 1.1,
    "axes.edgecolor": "black",
    "axes.facecolor": "white",
    "legend.fontsize": 12,
    "legend.frameon": False,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def style_full_box(ax, linewidth=1.1):
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(linewidth)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        width=1.0,
        length=5,
        color="black",
        top=False,
        right=False,
    )

    ax.set_axisbelow(True)
    ax.set_facecolor("white")


def save_curve_figure(fig, stem):
    png_path = CURVE_DIR / f"{stem}.png"
    pdf_path = CURVE_DIR / f"{stem}.pdf"

    fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


def make_curve_model():
    """
    Fresh model with the same architecture used during CV.
    Do not deepcopy final_model, because final_model belongs to the final
    train-on-all-train-pairs stage.
    """
    return ResiduePairGNN(
        node_dim=data.x.shape[1],
        edge_dim=data.edge_attr.shape[1],
        hidden_dim=CV_HIDDEN_DIM,
        heads=CV_HEADS,
        dropout=CV_DROPOUT,
    ).to(device)


def pairs_to_tensor_curve(pairs, tensor_device):
    pair_array = np.asarray(pairs, dtype=np.int64)

    if pair_array.ndim != 2 or pair_array.shape[1] != 2:
        raise ValueError(
            f"Expected pairs with shape [B, 2], received {pair_array.shape}"
        )

    return torch.as_tensor(pair_array, dtype=torch.long, device=tensor_device)


@torch.no_grad()
def predict_fold_probabilities(model, graph_data, pairs, batch_size=1024):
    """
    Predict class-1 probabilities for residue-pair list.
    """
    model.eval()

    pair_tensor = pairs_to_tensor_curve(pairs, graph_data.x.device)
    prob_chunks = []

    for start in range(0, len(pair_tensor), batch_size):
        batch_pairs = pair_tensor[start:start + batch_size]
        logits = model(graph_data, batch_pairs)
        probs = torch.sigmoid(logits)
        prob_chunks.append(probs.detach().cpu().numpy())

    return np.concatenate(prob_chunks)


def interpolate_pr_curve(recall, precision, recall_grid):
    """
    Interpolate PR curve safely.

    sklearn returns recall in descending order. We sort by recall and handle
    duplicate recall values by keeping the maximum precision at each recall.
    """
    pr_df = pd.DataFrame({
        "recall": recall,
        "precision": precision,
    })

    pr_df = (
        pr_df
        .groupby("recall", as_index=False)["precision"]
        .max()
        .sort_values("recall")
    )

    return np.interp(
        recall_grid,
        pr_df["recall"].values,
        pr_df["precision"].values,
    )


# ----------------------------------------------------------------------------
# 1. Safety checks
# ----------------------------------------------------------------------------
assert "best_fold_states" in globals(), (
    "best_fold_states not found. Run the 5-fold CV training cell first."
)

assert len(best_fold_states) == N_SPLITS, (
    f"Expected {N_SPLITS} saved fold states, found {len(best_fold_states)}."
)

assert len(train_pairs) == len(y_train_cls), (
    f"train_pairs length = {len(train_pairs)}, "
    f"y_train_cls length = {len(y_train_cls)}."
)

train_pairs_array = np.asarray(train_pairs, dtype=np.int64)
y_train_array = np.asarray(y_train_cls, dtype=int)

assert train_pairs_array.ndim == 2 and train_pairs_array.shape[1] == 2, (
    f"train_pairs should be [B, 2], got {train_pairs_array.shape}."
)

assert set(np.unique(y_train_array)).issubset({0, 1}), (
    f"Labels must be 0/1, got {np.unique(y_train_array)}."
)

data_for_curves = data.to(device)

# ----------------------------------------------------------------------------
# 2. Recreate the same StratifiedKFold splits used in CV
# ----------------------------------------------------------------------------
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_RANDOM_STATE,
)

fold_curve_rows = []
fold_metric_rows = []

roc_grid = np.linspace(0.0, 1.0, 500)
pr_recall_grid = np.linspace(0.0, 1.0, 500)

roc_tpr_interpolated = []
pr_precision_interpolated = []
positive_prevalence_values = []

print("=" * 80)
print("Generating 5-fold ROC and precision-recall curves")
print("=" * 80)

for fold_idx, (_, val_idx) in enumerate(
    skf.split(train_pairs_array, y_train_array),
    start=1,
):
    print(f"Fold {fold_idx}/{N_SPLITS}")

    val_pairs_fold = train_pairs_array[val_idx]
    y_val_fold = y_train_array[val_idx]

    assert len(np.unique(y_val_fold)) == 2, (
        f"Fold {fold_idx} validation set does not contain both classes."
    )

    fold_model = make_curve_model()
    fold_model.load_state_dict(best_fold_states[fold_idx - 1])
    fold_model.eval()

    y_prob_fold = predict_fold_probabilities(
        model=fold_model,
        graph_data=data_for_curves,
        pairs=val_pairs_fold,
        batch_size=CURVE_BATCH_SIZE,
    )

    # ROC
    fpr, tpr, roc_thresholds = roc_curve(y_val_fold, y_prob_fold)
    fold_auroc = roc_auc_score(y_val_fold, y_prob_fold)

    # Precision-recall
    precision, recall, pr_thresholds = precision_recall_curve(
        y_val_fold,
        y_prob_fold,
    )
    fold_auprc = average_precision_score(y_val_fold, y_prob_fold)

    fold_prevalence = float(np.mean(y_val_fold))
    positive_prevalence_values.append(fold_prevalence)

    fold_metric_rows.append({
        "fold": fold_idx,
        "n_validation_pairs": int(len(y_val_fold)),
        "n_positive": int(np.sum(y_val_fold == 1)),
        "n_negative": int(np.sum(y_val_fold == 0)),
        "positive_prevalence": fold_prevalence,
        "AUROC": float(fold_auroc),
        "AUPRC": float(fold_auprc),
    })

    # Save ROC points
    for x_val, y_val, th in zip(fpr, tpr, roc_thresholds):
        fold_curve_rows.append({
            "fold": fold_idx,
            "curve": "ROC",
            "x": float(x_val),
            "y": float(y_val),
            "threshold": float(th) if np.isfinite(th) else np.inf,
            "x_label": "False positive rate",
            "y_label": "True positive rate",
            "AUROC": float(fold_auroc),
            "AUPRC": float(fold_auprc),
            "positive_prevalence": fold_prevalence,
        })

    # Save PR points
    # PR thresholds has length len(precision)-1, so threshold is NaN for the last endpoint.
    pr_thresholds_full = list(pr_thresholds) + [np.nan]

    for x_val, y_val, th in zip(recall, precision, pr_thresholds_full):
        fold_curve_rows.append({
            "fold": fold_idx,
            "curve": "PR",
            "x": float(x_val),
            "y": float(y_val),
            "threshold": float(th) if np.isfinite(th) else np.nan,
            "x_label": "Recall",
            "y_label": "Precision",
            "AUROC": float(fold_auroc),
            "AUPRC": float(fold_auprc),
            "positive_prevalence": fold_prevalence,
        })

    # Interpolate ROC for mean curve
    interp_tpr = np.interp(roc_grid, fpr, tpr)
    interp_tpr[0] = 0.0
    interp_tpr[-1] = 1.0
    roc_tpr_interpolated.append(interp_tpr)

    # Interpolate PR for mean curve
    interp_precision = interpolate_pr_curve(
        recall=recall,
        precision=precision,
        recall_grid=pr_recall_grid,
    )
    pr_precision_interpolated.append(interp_precision)

    del fold_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


fold_curve_df = pd.DataFrame(fold_curve_rows)
fold_metric_df = pd.DataFrame(fold_metric_rows)

fold_curve_df.to_csv(
    CURVE_DIR / "cv_fold_roc_pr_curve_points.csv",
    index=False,
)

fold_metric_df.to_csv(
    CURVE_DIR / "cv_fold_curve_metrics.csv",
    index=False,
)

print("\nFold metrics:")
print(fold_metric_df)

mean_auroc = float(fold_metric_df["AUROC"].mean())
std_auroc = float(fold_metric_df["AUROC"].std(ddof=1))

mean_auprc = float(fold_metric_df["AUPRC"].mean())
std_auprc = float(fold_metric_df["AUPRC"].std(ddof=1))

mean_positive_prevalence = float(np.mean(positive_prevalence_values))

print("\nMean ± SD:")
print(f"AUROC = {mean_auroc:.4f} ± {std_auroc:.4f}")
print(f"AUPRC = {mean_auprc:.4f} ± {std_auprc:.4f}")
print(f"PR baseline prevalence = {mean_positive_prevalence:.4f}")

# Optional consistency check against fold_results_df if available
if "fold_results_df" in globals():
    if "AUROC" in fold_results_df.columns and "AUPRC" in fold_results_df.columns:
        print("\nConsistency check vs fold_results_df:")
        print(
            "  AUROC mean difference:",
            abs(mean_auroc - float(fold_results_df["AUROC"].mean())),
        )
        print(
            "  AUPRC mean difference:",
            abs(mean_auprc - float(fold_results_df["AUPRC"].mean())),
        )

# ----------------------------------------------------------------------------
# 3. Mean and SD curves
# ----------------------------------------------------------------------------
roc_tpr_interpolated = np.asarray(roc_tpr_interpolated)
pr_precision_interpolated = np.asarray(pr_precision_interpolated)

mean_tpr = roc_tpr_interpolated.mean(axis=0)
std_tpr = roc_tpr_interpolated.std(axis=0, ddof=1)

mean_precision = pr_precision_interpolated.mean(axis=0)
std_precision = pr_precision_interpolated.std(axis=0, ddof=1)

fold_colors = plt.get_cmap("tab10")(np.arange(N_SPLITS))

# ----------------------------------------------------------------------------
# 4. Combined ROC + PR figure
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14.2, 6.2))

ax_roc, ax_pr = axes

# -------------------------------
# ROC panel
# -------------------------------
for fold_idx in range(1, N_SPLITS + 1):
    sub = fold_curve_df[
        (fold_curve_df["fold"] == fold_idx)
        & (fold_curve_df["curve"] == "ROC")
    ]

    fold_auroc = fold_metric_df.loc[
        fold_metric_df["fold"] == fold_idx,
        "AUROC"
    ].iloc[0]

    ax_roc.plot(
        sub["x"],
        sub["y"],
        linewidth=1.8,
        alpha=0.88,
        color=fold_colors[fold_idx - 1],
        label=f"Fold {fold_idx} AUROC = {fold_auroc:.3f}",
    )

ax_roc.plot(
    roc_grid,
    mean_tpr,
    color="black",
    linewidth=2.8,
    label=f"Mean AUROC = {mean_auroc:.3f} ± {std_auroc:.3f}",
)

ax_roc.fill_between(
    roc_grid,
    np.maximum(mean_tpr - std_tpr, 0),
    np.minimum(mean_tpr + std_tpr, 1),
    color="black",
    alpha=0.10,
    linewidth=0,
)

ax_roc.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    color="gray",
    linewidth=1.2,
    label="Chance",
)

ax_roc.set_xlim(-0.01, 1.01)
ax_roc.set_ylim(-0.01, 1.01)
ax_roc.set_xlabel("False positive rate")
ax_roc.set_ylabel("True positive rate")
ax_roc.set_title("A  Five-fold ROC curves", loc="left", fontweight="bold", pad=10)
ax_roc.grid(linestyle="--", linewidth=0.6, alpha=0.25)
ax_roc.legend(loc="lower right", frameon=False, fontsize=11)
style_full_box(ax_roc)

# -------------------------------
# Precision-recall panel
# -------------------------------
for fold_idx in range(1, N_SPLITS + 1):
    sub = fold_curve_df[
        (fold_curve_df["fold"] == fold_idx)
        & (fold_curve_df["curve"] == "PR")
    ]

    fold_auprc = fold_metric_df.loc[
        fold_metric_df["fold"] == fold_idx,
        "AUPRC"
    ].iloc[0]

    ax_pr.plot(
        sub["x"],
        sub["y"],
        linewidth=1.8,
        alpha=0.88,
        color=fold_colors[fold_idx - 1],
        label=f"Fold {fold_idx} AUPRC = {fold_auprc:.3f}",
    )

ax_pr.plot(
    pr_recall_grid,
    mean_precision,
    color="black",
    linewidth=2.8,
    label=f"Mean AUPRC = {mean_auprc:.3f} ± {std_auprc:.3f}",
)

ax_pr.fill_between(
    pr_recall_grid,
    np.maximum(mean_precision - std_precision, 0),
    np.minimum(mean_precision + std_precision, 1),
    color="black",
    alpha=0.10,
    linewidth=0,
)

ax_pr.axhline(
    mean_positive_prevalence,
    linestyle="--",
    color="gray",
    linewidth=1.2,
    label=f"Baseline = {mean_positive_prevalence:.3f}",
)

ax_pr.set_xlim(-0.01, 1.01)
ax_pr.set_ylim(-0.01, 1.01)
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("B  Five-fold precision-recall curves", loc="left", fontweight="bold", pad=10)
ax_pr.grid(linestyle="--", linewidth=0.6, alpha=0.25)
ax_pr.legend(loc="lower left", frameon=False, fontsize=11)
style_full_box(ax_pr)

fig.tight_layout()

save_curve_figure(fig, "Figure_5fold_ROC_PR_curves")
plt.show()

# ----------------------------------------------------------------------------
# 5. Separate ROC-only figure
# ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.0))

for fold_idx in range(1, N_SPLITS + 1):
    sub = fold_curve_df[
        (fold_curve_df["fold"] == fold_idx)
        & (fold_curve_df["curve"] == "ROC")
    ]

    fold_auroc = fold_metric_df.loc[
        fold_metric_df["fold"] == fold_idx,
        "AUROC"
    ].iloc[0]

    ax.plot(
        sub["x"],
        sub["y"],
        linewidth=1.8,
        alpha=0.88,
        color=fold_colors[fold_idx - 1],
        label=f"Fold {fold_idx} AUROC = {fold_auroc:.3f}",
    )

ax.plot(
    roc_grid,
    mean_tpr,
    color="black",
    linewidth=2.8,
    label=f"Mean AUROC = {mean_auroc:.3f} ± {std_auroc:.3f}",
)

ax.fill_between(
    roc_grid,
    np.maximum(mean_tpr - std_tpr, 0),
    np.minimum(mean_tpr + std_tpr, 1),
    color="black",
    alpha=0.10,
    linewidth=0,
)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.2, label="Chance")

ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xlabel("False positive rate", fontsize=30)
ax.set_ylabel("True positive rate", fontsize=30)
ax.tick_params(axis="both", labelsize=28)
ax.grid(linestyle="--", linewidth=0.6, alpha=0.25)
ax.legend(loc="lower right", frameon=False, fontsize=16)

style_full_box(ax)
fig.tight_layout()

save_curve_figure(fig, "Figure_5fold_ROC_only")
plt.show()

# ----------------------------------------------------------------------------
# 6. Separate PR-only figure
# ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.0))

for fold_idx in range(1, N_SPLITS + 1):
    sub = fold_curve_df[
        (fold_curve_df["fold"] == fold_idx)
        & (fold_curve_df["curve"] == "PR")
    ]

    fold_auprc = fold_metric_df.loc[
        fold_metric_df["fold"] == fold_idx,
        "AUPRC"
    ].iloc[0]

    ax.plot(
        sub["x"],
        sub["y"],
        linewidth=1.8,
        alpha=0.88,
        color=fold_colors[fold_idx - 1],
        label=f"Fold {fold_idx} AUPRC = {fold_auprc:.3f}",
    )

ax.plot(
    pr_recall_grid,
    mean_precision,
    color="black",
    linewidth=2.8,
    label=f"Mean AUPRC = {mean_auprc:.3f} ± {std_auprc:.3f}",
)

ax.fill_between(
    pr_recall_grid,
    np.maximum(mean_precision - std_precision, 0),
    np.minimum(mean_precision + std_precision, 1),
    color="black",
    alpha=0.10,
    linewidth=0,
)

ax.axhline(
    mean_positive_prevalence,
    linestyle="--",
    color="gray",
    linewidth=1.2,
    label=f"Baseline = {mean_positive_prevalence:.3f}",
)

ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xlabel("Recall", fontsize=30)
ax.set_ylabel("Precision", fontsize=30)
ax.tick_params(axis="both", labelsize=28)
ax.grid(linestyle="--", linewidth=0.6, alpha=0.25)
ax.legend(loc="lower right", frameon=False, fontsize=14)

style_full_box(ax)
fig.tight_layout()

save_curve_figure(fig, "Figure_5fold_PR_only")
plt.show()

In [ ]:
# ============================================================================
# SELF-CONTAINED PUBLICATION-QUALITY LABEL-GUIDED UMAP
#
# This version fixes the error:
#   AssertionError: Run the previous UMAP embedding extraction cell first.
#
# It extracts learned pair embeddings from final_model first, then runs UMAP.
#
# Required variables:
#   final_model, data, train_pairs, y_train_cls, device
#
# Optional:
#   final_state, FINAL_BATCH_SIZE
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

try:
    import umap
except ImportError as e:
    raise ImportError(
        "UMAP is not installed. Install with: pip install umap-learn"
    ) from e

print("=" * 80)
print("Self-contained mildly label-guided UMAP")
print("=" * 80)

# ---------------------------------------------------------------------
# 1. Safety checks
# ---------------------------------------------------------------------
assert "final_model" in globals(), "final_model not found. Run final model training first."
assert "data" in globals(), "data not found."
assert "train_pairs" in globals(), "train_pairs not found."
assert "y_train_cls" in globals(), "y_train_cls not found."
assert "device" in globals(), "device not found."

# Load final trained weights if available
if "final_state" in globals():
    final_model.load_state_dict(final_state)
    print("Loaded final_state into final_model.")
else:
    print("WARNING: final_state not found; using current final_model weights.")

final_model.eval()
data = data.to(device)

out_dir = Path("umap_outputs")
out_dir.mkdir(exist_ok=True)

# ---------------------------------------------------------------------
# 2. Prepare train pairs and labels
# ---------------------------------------------------------------------
train_pairs_arr = np.asarray(train_pairs, dtype=np.int64)

# Accept both [B, 2] and [2, B]
if train_pairs_arr.ndim != 2:
    raise ValueError(f"train_pairs should be 2D, got shape {train_pairs_arr.shape}")

if train_pairs_arr.shape[1] == 2:
    pass
elif train_pairs_arr.shape[0] == 2:
    train_pairs_arr = train_pairs_arr.T
else:
    raise ValueError(f"train_pairs should be [B, 2] or [2, B], got {train_pairs_arr.shape}")

y_train_arr = np.asarray(y_train_cls, dtype=np.int64).reshape(-1)

assert len(train_pairs_arr) == len(y_train_arr), (
    f"train_pairs length {len(train_pairs_arr)} != y_train_cls length {len(y_train_arr)}"
)

assert set(np.unique(y_train_arr)).issubset({0, 1}), (
    f"Expected binary labels 0/1, got {np.unique(y_train_arr)}"
)

print(f"Training pairs : {len(train_pairs_arr)}")
print(f"Zero pairs     : {int((y_train_arr == 0).sum())}")
print(f"Nonzero pairs  : {int((y_train_arr == 1).sum())}")

# ---------------------------------------------------------------------
# 3. Local pair tensor helper
# ---------------------------------------------------------------------
def pairs_to_tensor_local(pair_array, tensor_device):
    pair_array = np.asarray(pair_array, dtype=np.int64)

    if pair_array.ndim != 2 or pair_array.shape[1] != 2:
        raise ValueError(f"Expected pair_array shape [B, 2], got {pair_array.shape}")

    return torch.as_tensor(pair_array, dtype=torch.long, device=tensor_device)

# ---------------------------------------------------------------------
# 4. Extract learned pair embeddings from the trained GNN
# ---------------------------------------------------------------------
@torch.no_grad()
def extract_pair_embeddings_for_umap(model, graph_data, pair_array, batch_size=512):
    """
    Extract learned pair embeddings from the trained GNN.

    Preferred embedding:
        output of classifier layers before the final logit layer.

    If classifier structure is not accessible, falls back to the symmetric
    pair features from model.make_pair_features().
    """
    model.eval()

    pair_tensor = pairs_to_tensor_local(pair_array, graph_data.x.device)

    # Compute node embeddings once
    if not hasattr(model, "encode_nodes"):
        raise AttributeError(
            "final_model does not have encode_nodes(). "
            "Use the ResiduePairGNN class version with encode_nodes and make_pair_features."
        )

    if not hasattr(model, "make_pair_features"):
        raise AttributeError(
            "final_model does not have make_pair_features(). "
            "Use the corrected symmetric ResiduePairGNN class."
        )

    h_nodes = model.encode_nodes(
        graph_data.x,
        graph_data.edge_index,
        graph_data.edge_attr
    )

    embedding_chunks = []
    logit_chunks = []

    for start in range(0, len(pair_tensor), batch_size):
        batch_pairs = pair_tensor[start:start + batch_size]

        pair_feat = model.make_pair_features(h_nodes, batch_pairs)

        # Generic extraction from nn.Sequential classifier:
        # all layers except final layer = embedding; final layer = logit.
        if hasattr(model, "classifier") and isinstance(model.classifier, torch.nn.Sequential):
            layers = list(model.classifier.children())

            if len(layers) >= 2:
                z = pair_feat

                for layer in layers[:-1]:
                    z = layer(z)

                logits = layers[-1](z).squeeze(-1)
                embedding = z
            else:
                logits = model(graph_data, batch_pairs)
                embedding = pair_feat

        else:
            logits = model(graph_data, batch_pairs)
            embedding = pair_feat

        embedding_chunks.append(embedding.detach().cpu().numpy())
        logit_chunks.append(logits.detach().cpu().numpy())

    pair_embeddings = np.concatenate(embedding_chunks, axis=0)
    logits = np.concatenate(logit_chunks, axis=0)

    probs = 1.0 / (1.0 + np.exp(-logits))

    return pair_embeddings, logits, probs

_umap_batch_size = FINAL_BATCH_SIZE if "FINAL_BATCH_SIZE" in globals() else 512

pair_embeddings, train_logits_umap, train_probs_umap = extract_pair_embeddings_for_umap(
    model=final_model,
    graph_data=data,
    pair_array=train_pairs_arr,
    batch_size=_umap_batch_size
)

print(f"Extracted pair embeddings: {pair_embeddings.shape}")
print(
    f"GNN probability range: "
    f"{train_probs_umap.min():.4f} – {train_probs_umap.max():.4f}"
)

# ---------------------------------------------------------------------
# 5. Standardize pair embeddings before UMAP
# ---------------------------------------------------------------------
scaler = StandardScaler()
pair_embeddings_scaled = scaler.fit_transform(pair_embeddings)

# ---------------------------------------------------------------------
# 6. Mildly supervised / label-guided UMAP
# ---------------------------------------------------------------------
# target_weight controls label influence.
# 0.0 = unsupervised
# 0.10-0.20 = mild label guidance
# >0.40 = too artificial unless explicitly stated.
TARGET_WEIGHT = 0.15

umap_label_guided = umap.UMAP(
    n_components=2,
    n_neighbors=18,
    min_dist=0.04,
    spread=1.2,
    metric="cosine",
    target_metric="categorical",
    target_weight=TARGET_WEIGHT,
    random_state=42,
    init="spectral",
)

umap_xy_guided = umap_label_guided.fit_transform(
    pair_embeddings_scaled,
    y=y_train_arr
)

umap_x_guided = umap_xy_guided[:, 0]
umap_y_guided = umap_xy_guided[:, 1]

if len(np.unique(y_train_arr)) == 2:
    sil_guided = silhouette_score(umap_xy_guided, y_train_arr)
else:
    sil_guided = np.nan

print(f"Label-guided UMAP silhouette score: {sil_guided:.4f}")
print(f"target_weight used: {TARGET_WEIGHT}")

# ---------------------------------------------------------------------
# 7. Save coordinates
# ---------------------------------------------------------------------
umap_df_guided = pd.DataFrame({
    "pair_i_0based": train_pairs_arr[:, 0],
    "pair_j_0based": train_pairs_arr[:, 1],
    "pair_i_1based": train_pairs_arr[:, 0] + 1,
    "pair_j_1based": train_pairs_arr[:, 1] + 1,
    "sequence_separation": np.abs(train_pairs_arr[:, 0] - train_pairs_arr[:, 1]),
    "label": y_train_arr,
    "label_name": np.where(y_train_arr == 1, "Nonzero pair", "Zero pair"),
    "gnn_logit": train_logits_umap,
    "gnn_probability_nonzero": train_probs_umap,
    "UMAP1_label_guided": umap_x_guided,
    "UMAP2_label_guided": umap_y_guided,
    "target_weight": TARGET_WEIGHT,
    "silhouette_score": sil_guided,
})

coord_path = out_dir / "train_pair_label_guided_umap_coordinates.csv"
umap_df_guided.to_csv(coord_path, index=False)

print("Saved:")
print(coord_path)

# ---------------------------------------------------------------------
# 8. Plotting style
# ---------------------------------------------------------------------
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 12,
    "axes.labelsize": 26,
    "axes.titlesize": 18,
    "axes.linewidth": 1.3,
    "xtick.labelsize": 22,
    "ytick.labelsize": 22,
    "legend.fontsize": 22,
    "legend.frameon": False,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def style_axes(ax):
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_linewidth(1)
        ax.spines[side].set_color("black")

    ax.tick_params(axis="both", direction="out", length=5, width=1.2)
    ax.grid(True, linestyle="--", linewidth=0.6, alpha=0.22)
    ax.set_facecolor("white")

# ---------------------------------------------------------------------
# 9. Plot label-guided UMAP by class
# ---------------------------------------------------------------------
zero_mask = y_train_arr == 0
pos_mask = y_train_arr == 1

fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(
    umap_x_guided[zero_mask],
    umap_y_guided[zero_mask],
    s=38,
    alpha=0.72,
    c="#4C78A8",
    edgecolors="white",
    linewidths=0.35,
    label=f"Zero pairs (n={int(zero_mask.sum())})",
)

ax.scatter(
    umap_x_guided[pos_mask],
    umap_y_guided[pos_mask],
    s=38,
    alpha=0.82,
    c="#E45756",
    edgecolors="white",
    linewidths=0.35,
    label=f"Nonzero pairs (n={int(pos_mask.sum())})",
)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.legend(loc="best")

style_axes(ax)
plt.tight_layout()

fig_path_png = out_dir / "Figure_train_pair_label_guided_UMAP_by_label.png"
fig_path_pdf = out_dir / "Figure_train_pair_label_guided_UMAP_by_label.pdf"

plt.savefig(fig_path_png, dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig(fig_path_pdf, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(fig_path_png)
print(fig_path_pdf)

# ---------------------------------------------------------------------
# 10. Optional probability-colored label-guided UMAP
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 7.0))

sc = ax.scatter(
    umap_x_guided,
    umap_y_guided,
    s=38,
    alpha=0.88,
    c=train_probs_umap,
    cmap="viridis",
    edgecolors="none",
)

cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("GNN probability of nonzero pair", fontsize=18)
cbar.ax.tick_params(labelsize=14)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

#style_axes(ax)
plt.tight_layout()

fig_prob_png = out_dir / "Figure_train_pair_label_guided_UMAP_by_probability.png"
fig_prob_pdf = out_dir / "Figure_train_pair_label_guided_UMAP_by_probability.pdf"

plt.savefig(fig_prob_png, dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig(fig_prob_pdf, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(fig_prob_png)
print(fig_prob_pdf)

print("\nDone.")

In [ ]:
# ============================================================================
# INTERPRETABLE SURROGATE SHAP FOR ResiduePairGNN
# Symmetric / order-invariant pair descriptors
#
# Purpose:
#   Train an interpretable random-forest surrogate to mimic the trained GNN's
#   predicted P(nonzero interaction), then explain the surrogate using TreeSHAP.
#
# Important:
#   This is a surrogate explanation of GNN behavior.
#   It is NOT direct attribution through the full GNN.
#
# Required variables:
#   final_model, final_state, data,
#   train_pairs, test_pairs,
#   y_train_cls, y_test_cls,
#   aa_order, device
# ============================================================================

import copy
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import shap
import torch

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# -----------------------------------------------------------------------------
# 0. Settings and output directory
# -----------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

SURROGATE_DIR = Path("interpretability_outputs/surrogate_shap")
SURROGATE_DIR.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.linewidth": 0.9,
    "axes.edgecolor": "black",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def style_full_box(ax, linewidth=0.9):
    for side in ("left", "right", "top", "bottom"):
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("black")
        ax.spines[side].set_linewidth(linewidth)

    ax.tick_params(
        axis="both",
        direction="out",
        width=0.8,
        length=4,
        top=False,
        right=False
    )

    ax.set_axisbelow(True)


def save_surrogate_figure(fig, filename_stem):
    fig.savefig(
        SURROGATE_DIR / f"{filename_stem}.pdf",
        bbox_inches="tight",
        facecolor="white"
    )
    fig.savefig(
        SURROGATE_DIR / f"{filename_stem}.png",
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )


def safe_corr(func, x, y):
    """
    Robust Pearson/Spearman helper.
    Returns nan if one vector is constant.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan, np.nan

    return func(x, y)


print("=" * 75)
print("Surrogate SHAP for ResiduePairGNN")
print("=" * 75)

# -----------------------------------------------------------------------------
# 1. Safety checks
# -----------------------------------------------------------------------------
assert "final_model" in globals(), "final_model not found. Run final training first."
assert "data" in globals(), "data not found."
assert "train_pairs" in globals(), "train_pairs not found."
assert "test_pairs" in globals(), "test_pairs not found."
assert "y_train_cls" in globals(), "y_train_cls not found."
assert "y_test_cls" in globals(), "y_test_cls not found."
assert "aa_order" in globals(), "aa_order not found."
assert "device" in globals(), "device not found."

# Load final trained weights if available
if "final_state" in globals():
    final_model.load_state_dict(final_state)
    print("Loaded final_state into final_model.")
else:
    print("WARNING: final_state not found. Using current final_model weights.")

final_model.eval()
data = data.to(device)

# -----------------------------------------------------------------------------
# 2. Use original 23 node features
# -----------------------------------------------------------------------------
node_features_np = data.x.detach().cpu().numpy().astype(np.float32)

print("\nOriginal node-feature matrix:")
print(f"  Shape: {node_features_np.shape}")

if node_features_np.shape[1] != 23:
    raise ValueError(
        f"This SHAP cell expects the original 23 node features, but data.x has "
        f"{node_features_np.shape[1]} features. If you added physicochemical "
        f"features, update pair_feature_table() and feature_names accordingly."
    )

# -----------------------------------------------------------------------------
# 3. Symmetric pair-feature table
# -----------------------------------------------------------------------------
def pair_feature_table(pairs):
    """
    Build order-invariant pair descriptors.

    Features:
        20 AA count features:
            one_hot_i + one_hot_j, values in {0, 1, 2}

        5 physicochemical/positional pair features:
            charge_i + charge_j
            charge_i * charge_j
            mean global position
            mean local fragment position
            sequence separation

    Total = 25 features.
    """
    pairs_arr = np.asarray(pairs, dtype=np.int64)

    if pairs_arr.ndim != 2 or pairs_arr.shape[1] != 2:
        raise ValueError(f"Expected pairs shape [B, 2], got {pairs_arr.shape}")

    rows = []
    sequence_denominator = max(1, node_features_np.shape[0] - 1)

    for residue_i, residue_j in pairs_arr:
        residue_i = int(residue_i)
        residue_j = int(residue_j)

        feat_i = node_features_np[residue_i]
        feat_j = node_features_np[residue_j]

        onehot_i, onehot_j = feat_i[:20], feat_j[:20]

        charge_i, charge_j = feat_i[20], feat_j[20]
        gpos_i, gpos_j = feat_i[21], feat_j[21]
        lpos_i, lpos_j = feat_i[22], feat_j[22]

        aa_count = onehot_i + onehot_j
        charge_sum = charge_i + charge_j
        charge_product = charge_i * charge_j
        mean_global_position = 0.5 * (gpos_i + gpos_j)
        mean_local_position = 0.5 * (lpos_i + lpos_j)
        sequence_separation = abs(residue_i - residue_j) / sequence_denominator

        row = np.concatenate([
            aa_count,
            np.asarray(
                [
                    charge_sum,
                    charge_product,
                    mean_global_position,
                    mean_local_position,
                    sequence_separation
                ],
                dtype=np.float32
            )
        ])

        rows.append(row)

    return np.asarray(rows, dtype=np.float32)


X_train_surrogate = pair_feature_table(train_pairs)
X_test_surrogate = pair_feature_table(test_pairs)

y_train_arr = np.asarray(y_train_cls, dtype=np.int64)
y_test_arr = np.asarray(y_test_cls, dtype=np.int64)

print("\nInterpretable pair-feature tables:")
print(f"  Train: {X_train_surrogate.shape}")
print(f"  Test : {X_test_surrogate.shape}")

# -----------------------------------------------------------------------------
# 4. Feature names
# -----------------------------------------------------------------------------
if len(aa_order) != 20:
    raise ValueError(f"Expected 20 amino-acid labels in aa_order, got {len(aa_order)}.")

feature_names = (
    [f"Pair contains {aa}" for aa in aa_order]
    + [
        "Pair charge sum",
        "Pair charge product",
        "Mean global residue position",
        "Mean local fragment position",
        "Sequence separation",
    ]
)

assert len(feature_names) == X_train_surrogate.shape[1], (
    f"{len(feature_names)} feature names for {X_train_surrogate.shape[1]} features."
)

# -----------------------------------------------------------------------------
# 5. Obtain GNN probabilities for train and test pairs
# -----------------------------------------------------------------------------
@torch.no_grad()
def gnn_predict_pairs(pairs, batch_size=1024):
    pairs_arr = np.asarray(pairs, dtype=np.int64)

    if pairs_arr.ndim != 2 or pairs_arr.shape[1] != 2:
        raise ValueError(f"Expected pairs shape [B, 2], got {pairs_arr.shape}")

    pair_tensor = torch.as_tensor(
        pairs_arr,
        dtype=torch.long,
        device=device
    )

    final_model.eval()

    prob_chunks = []

    for start in range(0, len(pair_tensor), batch_size):
        batch_pairs = pair_tensor[start:start + batch_size]
        probs = final_model.predict_proba(data, batch_pairs)
        prob_chunks.append(probs.detach().cpu().numpy())

    return np.concatenate(prob_chunks)


gnn_probs_train = gnn_predict_pairs(train_pairs)
gnn_probs_test = gnn_predict_pairs(test_pairs)

print("\nGNN probability ranges:")
print(f"  Train: {gnn_probs_train.min():.4f} – {gnn_probs_train.max():.4f}")
print(f"  Test : {gnn_probs_test.min():.4f} – {gnn_probs_test.max():.4f}")

# -----------------------------------------------------------------------------
# 6. Fit surrogate on training pairs only
# -----------------------------------------------------------------------------
surrogate = RandomForestRegressor(
    n_estimators=800,
    max_features="sqrt",
    min_samples_leaf=2,
    bootstrap=True,
    random_state=SEED,
    n_jobs=-1,
)

surrogate.fit(X_train_surrogate, gnn_probs_train)

surrogate_train_predictions = np.clip(
    surrogate.predict(X_train_surrogate),
    0.0,
    1.0
)

surrogate_test_predictions = np.clip(
    surrogate.predict(X_test_surrogate),
    0.0,
    1.0
)

# -----------------------------------------------------------------------------
# 7. Evaluate surrogate fidelity
# -----------------------------------------------------------------------------
train_r2 = r2_score(gnn_probs_train, surrogate_train_predictions)
test_r2 = r2_score(gnn_probs_test, surrogate_test_predictions)

train_mae = mean_absolute_error(gnn_probs_train, surrogate_train_predictions)
test_mae = mean_absolute_error(gnn_probs_test, surrogate_test_predictions)

test_rmse = np.sqrt(mean_squared_error(gnn_probs_test, surrogate_test_predictions))

test_pearson, test_pearson_p = safe_corr(
    pearsonr,
    gnn_probs_test,
    surrogate_test_predictions
)

test_spearman, test_spearman_p = safe_corr(
    spearmanr,
    gnn_probs_test,
    surrogate_test_predictions
)

fidelity_df = pd.DataFrame({
    "metric": [
        "Training R2",
        "Test R2",
        "Training MAE",
        "Test MAE",
        "Test RMSE",
        "Test Pearson correlation",
        "Test Spearman correlation",
        "Test Pearson p",
        "Test Spearman p",
    ],
    "value": [
        train_r2,
        test_r2,
        train_mae,
        test_mae,
        test_rmse,
        test_pearson,
        test_spearman,
        test_pearson_p,
        test_spearman_p,
    ],
})

fidelity_df.to_csv(
    SURROGATE_DIR / "surrogate_fidelity_metrics.csv",
    index=False
)

print("\n" + "=" * 75)
print("Surrogate fidelity to GNN probabilities")
print("=" * 75)
print(f"Training R²       : {train_r2:.4f}")
print(f"Test R²           : {test_r2:.4f}")
print(f"Training MAE      : {train_mae:.4f}")
print(f"Test MAE          : {test_mae:.4f}")
print(f"Test RMSE         : {test_rmse:.4f}")
print(f"Test Pearson r    : {test_pearson:.4f}")
print(f"Test Spearman rho : {test_spearman:.4f}")

if test_r2 < 0.70:
    print(
        "\nWARNING: Surrogate fidelity is weak on held-out test pairs. "
        "SHAP should be interpreted cautiously as an approximate summary "
        "of the surrogate, not the exact GNN."
    )
elif test_r2 < 0.90:
    print(
        "\nSurrogate fidelity is moderate. SHAP is useful as an approximate "
        "behavioral explanation of the GNN probability function."
    )
else:
    print("\nSurrogate fidelity is strong.")

# -----------------------------------------------------------------------------
# 8. Save prediction-level tables
# -----------------------------------------------------------------------------
surrogate_train_df = pd.DataFrame({
    "split": "train",
    "gnn_probability": gnn_probs_train,
    "surrogate_probability": surrogate_train_predictions,
    "true_label": y_train_arr,
})

surrogate_test_df = pd.DataFrame({
    "split": "test",
    "gnn_probability": gnn_probs_test,
    "surrogate_probability": surrogate_test_predictions,
    "true_label": y_test_arr,
})

surrogate_prediction_df = pd.concat(
    [surrogate_train_df, surrogate_test_df],
    ignore_index=True
)

surrogate_prediction_df.to_csv(
    SURROGATE_DIR / "surrogate_predictions_vs_gnn_probabilities.csv",
    index=False
)

# -----------------------------------------------------------------------------
# 9. Plot surrogate fidelity on held-out test pairs
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.8, 5.4))

ax.scatter(
    gnn_probs_test,
    surrogate_test_predictions,
    s=34,
    alpha=0.72,
    color="#4C78A8",
    edgecolors="white",
    linewidths=0.25,
    rasterized=True
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.2,
    color="black",
    label="Perfect agreement"
)

ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

ax.set_xlabel("GNN-predicted P(nonzero)")
ax.set_ylabel("Surrogate-predicted P(nonzero)")

ax.set_title(
    "Surrogate fidelity on held-out pairs",
    loc="left",
    fontweight="bold",
    pad=9
)

ax.text(
    0.04,
    0.96,
    f"$R^2$ = {test_r2:.3f}\nMAE = {test_mae:.3f}\nPearson $r$ = {test_pearson:.3f}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=9
)

ax.legend(frameon=False, loc="lower right")
ax.grid(linestyle="--", linewidth=0.5, alpha=0.22)
style_full_box(ax)

fig.tight_layout()
save_surrogate_figure(fig, "Figure_surrogate_fidelity")
plt.show()

# -----------------------------------------------------------------------------
# 10. TreeSHAP
# -----------------------------------------------------------------------------
# Recommended default:
#   Use held-out test SHAP if fidelity is acceptable.
#   For only 84 test samples, also save train SHAP for stability.
EXPLAIN_SPLIT = "test"  # options: "test" or "train"

if EXPLAIN_SPLIT == "test":
    X_explain = X_test_surrogate
    y_explain = y_test_arr
    split_name = "test"
elif EXPLAIN_SPLIT == "train":
    X_explain = X_train_surrogate
    y_explain = y_train_arr
    split_name = "train"
else:
    raise ValueError("EXPLAIN_SPLIT must be 'test' or 'train'.")

explainer = shap.TreeExplainer(surrogate)

shap_values = explainer.shap_values(
    X_explain,
    check_additivity=False
)

shap_values = np.asarray(shap_values)

if shap_values.ndim == 3:
    shap_values = shap_values[..., 0]

if shap_values.shape != X_explain.shape:
    raise ValueError(
        f"SHAP shape {shap_values.shape} does not match input shape {X_explain.shape}."
    )

np.save(
    SURROGATE_DIR / f"surrogate_SHAP_values_{split_name}.npy",
    shap_values
)

pd.DataFrame(
    shap_values,
    columns=feature_names
).to_csv(
    SURROGATE_DIR / f"surrogate_SHAP_values_{split_name}.csv",
    index=False
)

pd.DataFrame(
    X_explain,
    columns=feature_names
).to_csv(
    SURROGATE_DIR / f"surrogate_SHAP_input_features_{split_name}.csv",
    index=False
)

# -----------------------------------------------------------------------------
# 11. SHAP beeswarm
# -----------------------------------------------------------------------------
plt.figure()

shap.summary_plot(
    shap_values,
    X_explain,
    feature_names=feature_names,
    max_display=20,
    show=False,
    plot_size=(9.5, 7.0)
)

fig = plt.gcf()
ax = plt.gca()

ax.set_xlabel("SHAP value for surrogate P(nonzero)")
style_full_box(ax)

for figure_axis in fig.axes:
    for side in ("left", "right", "top", "bottom"):
        figure_axis.spines[side].set_visible(True)
        figure_axis.spines[side].set_color("black")
        figure_axis.spines[side].set_linewidth(0.8)

fig.tight_layout()
save_surrogate_figure(fig, f"Figure_surrogate_SHAP_beeswarm_{split_name}")
plt.show()

# -----------------------------------------------------------------------------
# 12. Mean absolute SHAP importance
# -----------------------------------------------------------------------------
mean_absolute_shap = np.mean(np.abs(shap_values), axis=0)

shap_importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_absolute_SHAP": mean_absolute_shap,
}).sort_values(
    "mean_absolute_SHAP",
    ascending=False
)

shap_importance_df.to_csv(
    SURROGATE_DIR / f"surrogate_SHAP_feature_importance_{split_name}.csv",
    index=False
)

TOP_SHAP_FEATURES = min(20, len(shap_importance_df))

shap_plot_df = (
    shap_importance_df
    .head(TOP_SHAP_FEATURES)
    .sort_values("mean_absolute_SHAP", ascending=True)
)

fig, ax = plt.subplots(figsize=(8.3, 6.5))

y_positions = np.arange(len(shap_plot_df))

ax.barh(
    y_positions,
    shap_plot_df["mean_absolute_SHAP"],
    color="#7A5195",
    edgecolor="black",
    linewidth=0.55
)

ax.set_yticks(y_positions)
ax.set_yticklabels(shap_plot_df["feature"], fontsize=10)
ax.set_xlabel("Mean absolute SHAP value")
ax.set_title(
    "Most influential interpretable surrogate features",
    loc="left",
    fontweight="bold",
    pad=9
)

ax.grid(axis="x", linestyle="--", linewidth=0.6, alpha=0.25)
style_full_box(ax)

fig.tight_layout()
save_surrogate_figure(fig, f"Figure_surrogate_SHAP_importance_{split_name}")
plt.show()

# -----------------------------------------------------------------------------
# 13. Save final summary
# -----------------------------------------------------------------------------
summary_txt = (
    "Surrogate SHAP analysis complete.\n"
    "This analysis explains a random-forest surrogate trained to mimic the "
    "GNN's predicted P(nonzero interaction) using symmetric interpretable "
    "pair descriptors. It is not direct attribution through the full GNN.\n"
    f"Explained split: {split_name}\n"
    f"Train R2: {train_r2:.4f}\n"
    f"Test R2: {test_r2:.4f}\n"
    f"Test MAE: {test_mae:.4f}\n"
    f"Test Pearson r: {test_pearson:.4f}\n"
    f"Test Spearman rho: {test_spearman:.4f}\n"
)

with open(SURROGATE_DIR / "surrogate_SHAP_summary.txt", "w") as f:
    f.write(summary_txt)

print("\n" + "=" * 75)
print("SURROGATE SHAP ANALYSIS COMPLETE")
print("=" * 75)
print(summary_txt)
print(f"Outputs saved in:\n{SURROGATE_DIR.resolve()}")

In [ ]:


from scipy.stats import pearsonr, spearmanr

 
out_dir = Path("gnn_full_interaction_outputs")
out_dir.mkdir(exist_ok=True)

GNN_THRESHOLD = 0.50

# ------------------------------------------------------------
# 1. Basic checks
# ------------------------------------------------------------
assert "N" in globals(), "N is not defined."
assert "all_edges" in globals(), "all_edges is not defined."
assert "positive_pairs" in globals(), "positive_pairs is not defined."
assert "zero_pairs" in globals(), "zero_pairs is not defined."

# All possible undirected residue pairs
all_possible_pairs = [
    (i, j)
    for i in range(N)
    for j in range(i + 1, N)
]

md_observed_pairs = set(positive_pairs) | set(zero_pairs)
unknown_pairs = sorted(set(all_possible_pairs) - md_observed_pairs)

print(f"MD-positive pairs     : {len(positive_pairs)}")
print(f"MD-zero pairs         : {len(zero_pairs)}")
print(f"MD-observed total     : {len(md_observed_pairs)}")
print(f"MD-unobserved pairs   : {len(unknown_pairs)}")

assert len(md_observed_pairs) == 1117
assert len(unknown_pairs) == 4136

# ------------------------------------------------------------
# 2. Build MD-only binary interaction matrix
# ------------------------------------------------------------
md_binary_matrix = np.zeros((N, N), dtype=np.int8)

for i, j in positive_pairs:
    md_binary_matrix[i, j] = 1
    md_binary_matrix[j, i] = 1

np.fill_diagonal(md_binary_matrix, 0)

# Number of true MD-positive partners for each residue
md_interaction_count = md_binary_matrix.sum(axis=1)

# ------------------------------------------------------------
# 3. Obtain GNN class-probability matrix
#
# Preferred matrix:
#   full_class_probability_matrix
#
# Expected meaning:
#   MD positive = 1
#   MD zero     = 0
#   unknown     = GNN P(nonzero)
# ------------------------------------------------------------
if "full_class_probability_matrix" in globals():

    gnn_probability_matrix = np.asarray(
        full_class_probability_matrix,
        dtype=float
    ).copy()

    probability_source = "memory: full_class_probability_matrix"

else:
    probability_path = (
        out_dir / "full_103x103_class_probability_matrix.csv"
    )

    if not probability_path.exists():
        raise FileNotFoundError(
            "The GNN class-probability matrix was not found.\n"
            "Run the cell that creates full_class_probability_matrix first.\n"
            "The raw-like MD+GNN matrix cannot be used reliably for "
            "binary interaction counting."
        )

    gnn_probability_matrix = pd.read_csv(
        probability_path,
        index_col=0
    ).values

    probability_source = str(probability_path)

assert gnn_probability_matrix.shape == (N, N)
assert np.allclose(
    gnn_probability_matrix,
    gnn_probability_matrix.T,
    atol=1e-7,
    equal_nan=True
), "GNN probability matrix is not symmetric."

print(f"GNN probability source: {probability_source}")

# ------------------------------------------------------------
# 4. Build GNN-only predicted binary matrix for unknown pairs
# ------------------------------------------------------------
gnn_only_binary_matrix = np.zeros((N, N), dtype=np.int8)

for i, j in unknown_pairs:
    predicted_contact = (
        gnn_probability_matrix[i, j] >= GNN_THRESHOLD
    )

    if predicted_contact:
        gnn_only_binary_matrix[i, j] = 1
        gnn_only_binary_matrix[j, i] = 1

np.fill_diagonal(gnn_only_binary_matrix, 0)

# Number of newly predicted partners for each residue
gnn_only_interaction_count = gnn_only_binary_matrix.sum(axis=1)

# ------------------------------------------------------------
# 5. Build hybrid MD + GNN binary interaction matrix
# ------------------------------------------------------------
hybrid_binary_matrix = np.maximum(
    md_binary_matrix,
    gnn_only_binary_matrix
)

np.fill_diagonal(hybrid_binary_matrix, 0)

hybrid_interaction_count = hybrid_binary_matrix.sum(axis=1)

# Sanity check:
# hybrid count = MD count + newly predicted unknown count
assert np.array_equal(
    hybrid_interaction_count,
    md_interaction_count + gnn_only_interaction_count
)

# ------------------------------------------------------------
# 6. Correlations across the 103 residues
# ------------------------------------------------------------
pearson_md_hybrid_r, pearson_md_hybrid_p = pearsonr(
    md_interaction_count,
    hybrid_interaction_count
)

spearman_md_hybrid_r, spearman_md_hybrid_p = spearmanr(
    md_interaction_count,
    hybrid_interaction_count
)

pearson_md_gnn_r, pearson_md_gnn_p = pearsonr(
    md_interaction_count,
    gnn_only_interaction_count
)

spearman_md_gnn_r, spearman_md_gnn_p = spearmanr(
    md_interaction_count,
    gnn_only_interaction_count
)

print(f"\nGNN threshold: {GNN_THRESHOLD:.2f}")

print("\nMD-only count versus hybrid count:")
print(
    f"  Pearson  r = {pearson_md_hybrid_r:.4f}, "
    f"p = {pearson_md_hybrid_p:.4e}"
)
print(
    f"  Spearman ρ = {spearman_md_hybrid_r:.4f}, "
    f"p = {spearman_md_hybrid_p:.4e}"
)

print("\nMD-only count versus newly predicted GNN-only count:")
print(
    f"  Pearson  r = {pearson_md_gnn_r:.4f}, "
    f"p = {pearson_md_gnn_p:.4e}"
)
print(
    f"  Spearman ρ = {spearman_md_gnn_r:.4f}, "
    f"p = {spearman_md_gnn_p:.4e}"
)

# ------------------------------------------------------------
# 7. Residue-level results table
# ------------------------------------------------------------
residue_numbers = np.arange(1, N + 1)

if "residue_letters" in globals():
    residue_aa = list(residue_letters)
else:
    residue_aa = [""] * N

if "residue_labels" in globals():
    residue_names = list(residue_labels)
else:
    residue_names = [
        f"{aa}{idx}"
        for idx, aa in zip(residue_numbers, residue_aa)
    ]

# ------------------------------------------------------------
# 7. Residue-level results table
# ------------------------------------------------------------
residue_numbers = np.arange(1, N + 1)

if "residue_letters" in globals():
    residue_aa = list(residue_letters)
else:
    residue_aa = [""] * N

if "residue_labels" in globals():
    residue_names = list(residue_labels)
else:
    residue_names = [
        f"{aa}{idx}"
        for idx, aa in zip(residue_numbers, residue_aa)
    ]

# ------------------------------------------------------------
# Calculate GNN-only / MD interaction-count multiplier

# ------------------------------------------------------------
gnn_to_md_multiplier = np.divide(
    gnn_only_interaction_count.astype(float),
    md_interaction_count.astype(float),
    out=np.full(
        md_interaction_count.shape,
        np.nan,
        dtype=float
    ),
    where=md_interaction_count > 0
)

count_df = pd.DataFrame({

    "Residue": residue_names,

    "MD_interaction_count":
        md_interaction_count.astype(int),

    "GNN_new_interaction_count":
        gnn_only_interaction_count.astype(int),

    # New column:
    # GNN-only predicted count divided by MD count
    "GNN_to_MD_multiplier":
        np.round(gnn_to_md_multiplier, 0),

   
})

count_path = (
    out_dir /
    "residue_interaction_counts_MD_vs_MD_plus_GNN.csv"
)

count_df.to_csv(
    count_path,
    index=False
)

print("\nResidue-level interaction counts:")
print(
    count_df.head(103).to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# CELL — Correlation between MD interaction counts and
#        newly predicted GNN interaction counts
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ------------------------------------------------------------
# 1. Extract residue-level counts
# ------------------------------------------------------------
md_count = count_df[
    "MD_interaction_count"
].to_numpy(dtype=float)

gnn_count = count_df[
    "GNN_new_interaction_count"
].to_numpy(dtype=float)

residue_labels_plot = count_df[
    "Residue"
].astype(str).to_numpy()

# ------------------------------------------------------------
# 2. Calculate correlations
# ------------------------------------------------------------
pearson_r, pearson_p = pearsonr(
    md_count,
    gnn_count
)

spearman_rho, spearman_p = spearmanr(
    md_count,
    gnn_count
)

print("=" * 70)
print("MD interaction count versus new GNN interaction count")
print("=" * 70)

print(
    f"Pearson r   = {pearson_r:.4f} | "
    f"p = {pearson_p:.4e}"
)

print(
    f"Spearman ρ  = {spearman_rho:.4f} | "
    f"p = {spearman_p:.4e}"
)

# ------------------------------------------------------------
# 3. Scatter plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    md_count,
    gnn_count,
    s=65,
    alpha=0.75
)

# Linear trend line
slope, intercept = np.polyfit(
    md_count,
    gnn_count,
    1
)

x_line = np.linspace(
    md_count.min(),
    md_count.max(),
    200
)

y_line = slope * x_line + intercept

ax.plot(
    x_line,
    y_line,
    linewidth=2
)

ax.set_xlabel(
    "Number of MD-observed interactions per residue",
    fontsize=14
)

ax.set_ylabel(
    "Number of newly predicted GNN interactions per residue",
    fontsize=14
)

 

ax.text(
    0.04,
    0.96,
    (
        f"Pearson r = {pearson_r:.3f}\n"
        
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=12
)

ax.tick_params(
    axis="both",
    labelsize=12
)

plt.tight_layout()

correlation_path = (
    out_dir /
    "MD_count_vs_GNN_new_count_correlation.png"
)

plt.savefig(
    correlation_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(f"\nSaved: {correlation_path}")


In [ ]:
# ------------------------------------------------------------
# Plot contact counts per residue: MD versus GNN
# ------------------------------------------------------------

xpos = np.arange(N)

# Use residue labels such as A1, G2, etc.
if "residue_labels" in globals():
    xlabels = residue_labels
else:
    xlabels = [str(i + 1) for i in range(N)]

fig, ax = plt.subplots(figsize=(40, 8))

# Intra-fragment contacts observed by MD
ax.plot(
    xpos,
    md_interaction_count,
    color="#2C7BB6",
    linewidth=3.5,
    marker="o",
    markersize=6,
    label="MD only",
    zorder=3
)

# Inter-fragment contacts predicted by GNN
ax.plot(
    xpos,
    gnn_only_interaction_count,
    color="#E87500",
    linewidth=3.5,
    marker="o",
    markersize=6,
    label=f"GNN only",
    zorder=4
)

ax.set_ylabel("Number of contacts", fontsize=30)

ax.set_xticks(xpos)
ax.set_xticklabels(
    xlabels,
    rotation=90,
    fontsize=18
)

ax.tick_params(axis="y", labelsize=24)
ax.set_xlim(-1, N)

ax.grid(
    axis="y",
    color="gray",
    linestyle="--",
    linewidth=1,
    alpha=0.30
)

ax.legend(
    fontsize=22,
    frameon=True,
    edgecolor="black",
    loc="upper right"
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(3)

plt.tight_layout()

plot_path = out_dir / "MD_vs_GNN_contact_count_per_residue.png"

plt.savefig(
    plot_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {plot_path}")

In [ ]:
# ------------------------------------------------------------
# Plot MD and GNN contact counts with baseline and multiplier
# ------------------------------------------------------------

MD_BASELINE = 5

xpos = np.arange(N)

if "residue_labels" in globals():
    xlabels = residue_labels
else:
    xlabels = [str(i + 1) for i in range(N)]

# Residues included in multiplier calculation
above_baseline = md_interaction_count >= MD_BASELINE

# Calculate GNN/MD multiplier for those residues
multipliers = (
    gnn_only_interaction_count[above_baseline].astype(float) /
    md_interaction_count[above_baseline].astype(float)
)

# mean is less sensitive to extreme residues than the mean
mean_multiplier = np.mean(multipliers)

# Expected GNN count if GNN simply multiplies MD by one number
fixed_multiplier_count = (
    mean_multiplier * md_interaction_count.astype(float)
)

print(f"MD baseline                       : {MD_BASELINE}")
print(f"Residues at or above baseline     : {above_baseline.sum()}")
print(f"Mean GNN/MD multiplier          : {mean_multiplier:.2f}")

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(40, 8))

# MD contact count
ax.plot(
    xpos,
    md_interaction_count,
    color="#2C7BB6",
    linewidth=3.5,
    marker="o",
    markersize=6,
    label="MD-only contacts",
    zorder=4
)

# Actual GNN-predicted contact count
ax.plot(
    xpos,
    gnn_only_interaction_count,
    color="#E87500",
    linewidth=3.5,
    marker="o",
    markersize=6,
    label="GNN-only predicted contacts",
    zorder=5
)

# Horizontal MD baseline
ax.axhline(
    y=MD_BASELINE,
    color="black",
    linewidth=3,
    linestyle="--",
    label=f"MD baseline = {MD_BASELINE}",
    zorder=2
)

# Expected curve under a fixed-multiplier explanation
ax.plot(
    xpos,
    fixed_multiplier_count,
    color="#2E8B57",
    linewidth=3,
    linestyle="--",
    label=(
        f"Fixed-multiplier reference: "
        f"{mean_multiplier:.2f} × MD"
    ),
    zorder=3
)

ax.set_ylabel(
    "Number of contacts",
    fontsize=30
)

ax.set_xticks(xpos)
ax.set_xticklabels(
    xlabels,
    rotation=90,
    fontsize=18
)

ax.tick_params(
    axis="y",
    labelsize=24
)

ax.set_xlim(-1, N)

ax.grid(
    axis="y",
    color="gray",
    linestyle="--",
    linewidth=1,
    alpha=0.30
)

ax.legend(
    fontsize=30,
    frameon=True,
    edgecolor="black"
)

#ax.set_ylim(-4, 30)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(3)

plt.tight_layout()

plot_path = (
    out_dir /
    f"MD_vs_GNN_contacts_with_baseline_{MD_BASELINE}.png"
)

plt.savefig(
    plot_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"Saved: {plot_path}")